# Heterogeneous Diffusion MoE (HDMOEM v3) — Colab A100 Nature-50k / COCO Training & Diagnostics (`text_guided`)

This notebook runs the training and diagnostic pipeline on the `text_guided` branch at **`256×256` RGB (`32×32×4` VAE latent)** resolution with **in-notebook architecture, optimization, loss-calibration, blend-anchor, router-masking, and dataset fixes** (zero edits to tracked repo `.py` files).

### Active Architecture, Training & Post-8k Option-3 Upgrades (`256×256` -> `32×32×4`)
* **Fix 1–6 Core:** Spherical blend (`HDMOEM_v3`), orthogonal frequency branch losses (`EDM_LOSS_NOD`), normalized MP weights + scalar gates, shared Mixture-CDF routers (`transition_sigma = 2.5`, `transition_sharpness = 0.8`), and `PooledCrossAttn` (`16×16` K/V grid).
* **Full-Res `3×3` ViT Spatial Continuity + `[1, 2]` Hybrid Fix:** Upgrades `HDMOEM_v3.output_proj_vit` from pointwise `1×1` to `3×3` `MP_Conv` (+~4k params) to eliminate `2×2` PixelShuffle latent seam artifacts at `32×32`, and fixes `hybrid_300m_p12` patch ordering (`[1, 2]`) so fine-detail Expert 0 ($\sigma < 0.5$) runs at full `32×32` resolution.
* **Balanced Frequency Split & Loss Reweighting:** Tightens `laplacian_std` from `1.0` -> `0.5` (`21.6%` low / `78.4%` high latent spectral energy split vs `5.6% / 94.4%`) and reweights `alpha_blend = 0.8`, `alpha_unet = 0.1`, `alpha_vit = 0.1` so optimizer gradients prioritize full-spectrum blended denoising.
* **WSD LR Schedule + Higher CFG Dropout + Calibrated Targets:** Replaces immediate cosine decay with **Warmup-Stable-Decay (WSD)** (`5%` warmup -> flat `1.0×` plateau to `85%` -> cosine decay to `0.10×`), raises `cfg_drop_prob` to `0.15`, calibrates the `sigma_data = 0.8` MSE target (`< 0.20`, beating the `~0.204` $1/f^2$ linear prior), and sets default evaluation CFG to `3.0` (`[1.0, 3.0, 5.0]` sweep).
* **Blend Handoff Anchor & Router Masks:** Bounds `HDMOEM_v3.compute_blend_weights` residual shift to `±0.4` (`blend_anchor_weight = 0.02`), passes `MaskGenerator` masks into `EDM_Sampler` & trajectory probes, fixes Plotter labels and Adam unit-RMS update telemetry, and logs a fixed-$\sigma$ 16-image `val_probe_mse`.
* **~50k Complex Nature Scenes Dataset (`mertcobanov/nature-dataset`):** Defaults to 50,000 real captioned nature/landscape images (48k train / 2k val, ~2.6 GB Parquet download) alongside MS-COCO 2017 (`coco2017_nature_50k`, `coco2017_full`).


In [ ]:
# @title 1. Runtime Check, Drive Mount, Clone `text_guided`, & Run Fix Verification Tests
import importlib.metadata
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
REPO_URL = "https://github.com/cs2mosa/Heterogeneous-MOE-for-Diffusion-models.git"
BRANCH_NAME = "text_guided"
REPO_DIR = Path("/content/Heterogeneous-MOE-for-Diffusion-models")
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/HDMOEM_colab")
RUN_NAME = "hdmoem_v3_nature50k_256_v3"  # @param {type:"string"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RUN_DIR = DRIVE_BASE_DIR / RUN_NAME
    DRIVE_DATA_CACHE = DRIVE_BASE_DIR / "datasets" / "coco2017"
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_DATA_CACHE.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_RUN_DIR = Path(f"/content/runs/{RUN_NAME}")
    DRIVE_DATA_CACHE = Path("/content/data_cache/coco2017")
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_DATA_CACHE.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {gpu_name} | Total VRAM: {vram_gb:.2f} GB | BF16 supported: {torch.cuda.is_bf16_supported()}")
    if vram_gb < 45:
        print("Detected ~40GB GPU -> Default recommendation: MICRO_BATCH_SIZE = 8 or 16 with grad accumulation.")
    else:
        print("Detected ~80GB GPU -> Default recommendation: MICRO_BATCH_SIZE = 16 or 32 with grad accumulation.")
else:
    raise RuntimeError("CUDA GPU not detected. Select an A100 GPU runtime in Colab.")

# Clone or update the exact branch
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "-b", BRANCH_NAME, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH_NAME], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH_NAME], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH_NAME], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("\nGit status / HEAD commit:")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

# Install only missing dependencies without disturbing Colab's pinned pandas/scipy/torch
if shutil.which("aria2c") is None:
    print("Installing aria2 for resilient multi-stream COCO downloads...")
    subprocess.run(
        "apt-get -qq update && apt-get -qq install -y aria2",
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False,
    )

missing_pkgs = []
for mod_name, pkg_spec in [
    ("diffusers", "diffusers"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("datasets", "datasets>=2.18.0"),
    ("huggingface_hub", "huggingface-hub>=0.25.0,<2.0"),
    ("safetensors", "safetensors"),
    ("seaborn", "seaborn"),
    ("pytest", "pytest"),
    ("cleanfid", "clean-fid"),
]:
    if importlib.util.find_spec(mod_name) is None:
        missing_pkgs.append(pkg_spec)
if missing_pkgs:
    print(f"Installing missing packages: {missing_pkgs}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)

# Repair Colab-incompatible upgrades if an earlier command bumped pandas>=3 or huggingface-hub>=2
repairs = []
try:
    if int(importlib.metadata.version("huggingface-hub").split(".")[0]) >= 2:
        repairs.append("huggingface-hub>=0.25.0,<2.0")
except importlib.metadata.PackageNotFoundError:
    pass
try:
    if int(importlib.metadata.version("pandas").split(".")[0]) >= 3:
        repairs.append("pandas==2.2.3")
except importlib.metadata.PackageNotFoundError:
    pass
if repairs:
    print(f"Restoring Colab-compatible pins before imports: {repairs}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *repairs], check=True)

# Verify Fixes 1-6 pass on this cloned commit before allocating GPU credits to training
print("\nRunning repository regression tests...")
_env = {**os.environ, "PYTHONPATH": str(REPO_DIR)}
subprocess.run([
    sys.executable, "-m", "pytest",
    "tests/files",
    "tests/test_model",
    "tests/test_utilities/test_mask_generator.py",
    "tests/test_utilities/test_sampler.py",
    "-q",
], check=True, env=_env)
print("All repository checks passed.")


## 2. Complex Scenes Dataset Setup (`mertcobanov/nature-dataset` ~50k or MS-COCO 2017)
Supports three dataset modes via `DATASET_MODE`:
* **`"nature_50k"` (Default — Recommended):** Loads **`mertcobanov/nature-dataset`** from Hugging Face (`~50,000` captioned nature/landscape photographs stored as a single ~2.61 GB Parquet dataset). Holds out `VAL_SIZE = 2,000` validation images (`48,000` train / `2,000` val, seed `42`). Avoids COCO's indoor object clutter while downloading ~7× faster than `train2017.zip`.
* **`"coco2017_nature_50k"`:** Downloads MS-COCO 2017 and filters to outdoor/nature/landscape scenes up to 50,000 training images.
* **`"coco2017_full"`:** Full MS-COCO 2017 (`train2017` ~118k images + `val2017` 5k images) using all 5 captions per image.


In [ ]:
# @title 2. Load 50k Complex Nature Scenes (`mertcobanov/nature-dataset`) or MS-COCO 2017
import io
import json
import os
import random
import re
import shutil
import subprocess
import zipfile
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

DATASET_MODE = "nature_50k"  # @param ["nature_50k", "coco2017_nature_50k", "coco2017_full"]
NATURE_HF_REPO = "mertcobanov/nature-dataset"
NATURE_VAL_SIZE = 2000  # @param {type:"integer"}
DATASET_SEED = 42
CACHE_NATURE_TO_DRIVE = False  # @param {type:"boolean"}

DATA_ROOT = Path("/content/coco2017")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
NATURE_CACHE_DIR = (
    (DRIVE_DATA_CACHE.parent / "hf_nature_50k")
    if CACHE_NATURE_TO_DRIVE
    else Path("/content/hf_nature_50k")
)

DOWNLOAD_TRAIN2017 = True  # @param {type:"boolean"}
CACHE_ZIPS_TO_DRIVE = False  # @param {type:"boolean"}  # Set True for coco2017_* if you have >=25GB free on Drive
USE_HF_MIRROR_FIRST = True  # @param {type:"boolean"}
HF_MIRROR_REPO = "manh6054/MSCOCO"
OFFICIAL_COCO_URLS = {
    "annotations_trainval2017.zip": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    "val2017.zip": "http://images.cocodataset.org/zips/val2017.zip",
    "train2017.zip": "http://images.cocodataset.org/zips/train2017.zip",
}

NATURE_INCLUDE_RE = re.compile(
    r"\b(?:mountain|mountains|forest|woods|tree|trees|lake|river|stream|waterfall|"
    r"ocean|sea|beach|shore|coast|cliff|valley|canyon|desert|dune|meadow|field|"
    r"grassland|hill|hills|island|glacier|snowy|sunset|sunrise|cloud|clouds|sky|"
    r"horizon|landscape|scenic|nature|wilderness|pond|wetland|marsh|volcano|rocky)\b",
    re.IGNORECASE,
)
NATURE_EXCLUDE_RE = re.compile(
    r"\b(?:kitchen|toilet|bathroom|sink|microwave|refrigerator|oven|couch|sofa|"
    r"bed|bedroom|laptop|computer|keyboard|television|tv|monitor|desk|office|"
    r"pizza|sandwich|cake|plate|fork|knife|toilet|urinal|bus|subway|train station)\b",
    re.IGNORECASE,
)


def make_image_transform(img_res: int = 256):
    return transforms.Compose([
        transforms.Resize(img_res, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(img_res),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])


def _coerce_pil_rgb(raw_img) -> Image.Image:
    if isinstance(raw_img, Image.Image):
        return raw_img.convert("RGB")
    if isinstance(raw_img, dict):
        if raw_img.get("bytes") is not None:
            return Image.open(io.BytesIO(raw_img["bytes"])).convert("RGB")
        if raw_img.get("path"):
            with Image.open(raw_img["path"]) as im:
                return im.convert("RGB")
    if isinstance(raw_img, (bytes, bytearray)):
        return Image.open(io.BytesIO(raw_img)).convert("RGB")
    return Image.open(raw_img).convert("RGB")


class HFImageCaptionDataset(Dataset):
    '''Wraps a HuggingFace image-caption Dataset into float32 images in [-1, 1] + prompt strings.'''
    def __init__(self, hf_ds, img_col: str, cap_col: str, img_res: int = 256):
        self.hf_ds = hf_ds
        self.img_col = img_col
        self.cap_col = cap_col
        self.transform = make_image_transform(img_res)

    def __len__(self):
        return len(self.hf_ds)

    def __getitem__(self, idx):
        n = len(self.hf_ds)
        for attempt in range(5):
            cur = (idx + attempt) % n
            try:
                row = self.hf_ds[cur]
                img = _coerce_pil_rgb(row[self.img_col])
                prompt = str(row.get(self.cap_col, "") or "").strip()
                if not prompt:
                    prompt = "a natural landscape photograph"
                return {"pixel_values": self.transform(img), "prompts": prompt}
            except Exception:
                if attempt == 4:
                    raise


def load_hf_nature_50k(repo_id: str, val_size: int, seed: int, cache_dir: Path, img_res: int = 256):
    from datasets import Image as HFImage, load_dataset

    cache_dir.mkdir(parents=True, exist_ok=True)
    print(f"[HF Dataset] Loading {repo_id} (cache -> {cache_dir})...")
    ds_dict = load_dataset(repo_id, cache_dir=str(cache_dir))
    full_ds = ds_dict["train"] if "train" in ds_dict else next(iter(ds_dict.values()))

    cols = list(full_ds.column_names)
    img_col = "image" if "image" in cols else next(
        (c for c, feat in full_ds.features.items() if isinstance(feat, HFImage)),
        cols[0],
    )
    cap_col = next(
        (c for c in ("caption", "text", "prompt", "description") if c in cols),
        next((c for c in cols if c != img_col), cols[-1]),
    )
    print(f"[HF Dataset] Columns: {cols} -> using image='{img_col}', caption='{cap_col}' | rows={len(full_ds):,}")

    if "validation" in ds_dict or "val" in ds_dict or "test" in ds_dict:
        val_key = next(k for k in ("validation", "val", "test") if k in ds_dict)
        train_hf = full_ds
        val_hf = ds_dict[val_key]
    else:
        val_count = min(int(val_size), max(1, len(full_ds) // 10))
        split = full_ds.train_test_split(test_size=val_count, seed=seed, shuffle=True)
        train_hf, val_hf = split["train"], split["test"]

    train_ds = HFImageCaptionDataset(train_hf, img_col=img_col, cap_col=cap_col, img_res=img_res)
    val_ds = HFImageCaptionDataset(val_hf, img_col=img_col, cap_col=cap_col, img_res=img_res)
    print(f"[HF Dataset] Split ready -> train={len(train_ds):,} | val={len(val_ds):,}")
    return train_ds, val_ds


def ensure_coco_archive(zip_name: str, expected_dir: Path):
    if expected_dir.exists() and any(expected_dir.iterdir()):
        print(f"[OK] Found extracted folder: {expected_dir}")
        return

    local_zip = DATA_ROOT / zip_name
    drive_zip = DRIVE_DATA_CACHE / zip_name

    if not local_zip.exists():
        if drive_zip.exists():
            print(f"[Drive Cache] Copying {zip_name} to local NVMe...")
            shutil.copy2(drive_zip, local_zip)
        elif USE_HF_MIRROR_FIRST and zip_name in ("train2017.zip", "val2017.zip"):
            try:
                from huggingface_hub import hf_hub_download
                print(f"[HF Mirror] Downloading {zip_name} from {HF_MIRROR_REPO}...")
                downloaded = hf_hub_download(
                    repo_id=HF_MIRROR_REPO,
                    filename=zip_name,
                    repo_type="dataset",
                    local_dir=str(DATA_ROOT),
                )
                if Path(downloaded) != local_zip and Path(downloaded).exists():
                    shutil.move(downloaded, local_zip)
            except Exception as e:
                print(f"[HF Mirror] Unavailable ({e}), falling back to official COCO HTTP via aria2c...")

    if not local_zip.exists():
        url = OFFICIAL_COCO_URLS[zip_name]
        print(f"[aria2c] Downloading {zip_name} from {url}...")
        subprocess.run([
            "aria2c", "-c", "-x", "16", "-s", "16", "--min-split-size=8M",
            "-d", str(DATA_ROOT), "-o", zip_name, url
        ], check=True)

    if CACHE_ZIPS_TO_DRIVE and not drive_zip.exists():
        print(f"[Drive Cache] Saving {zip_name} to Google Drive for future sessions...")
        shutil.copy2(local_zip, drive_zip)

    print(f"[Extract] Unzipping {zip_name} -> {DATA_ROOT}...")
    with zipfile.ZipFile(local_zip, "r") as zf:
        zf.extractall(DATA_ROOT)
    # Remove local zip after extraction to preserve /content disk headroom
    if local_zip.exists():
        local_zip.unlink()


def build_coco_records(img_dir: Path, ann_file: Path, max_samples: int = None, nature_only: bool = False):
    with open(ann_file, "r", encoding="utf-8") as f:
        coco = json.load(f)

    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    id_to_caps = defaultdict(list)
    for ann in coco["annotations"]:
        cap = ann.get("caption", "").strip()
        if cap:
            id_to_caps[ann["image_id"]].append(cap)

    records = []
    missing = 0
    for img_id, fname in id_to_file.items():
        caps = id_to_caps.get(img_id)
        if not caps:
            continue
        if nature_only:
            joined = " ".join(caps)
            if NATURE_EXCLUDE_RE.search(joined):
                continue
            nature_caps = [c for c in caps if NATURE_INCLUDE_RE.search(c)]
            if not nature_caps:
                continue
            caps = nature_caps
        p = img_dir / fname
        if p.exists():
            records.append((str(p), caps))
        else:
            missing += 1
        if max_samples is not None and len(records) >= max_samples:
            break

    tag = "nature-filtered " if nature_only else ""
    print(f"Indexed {len(records):,} {tag}images from {img_dir.name} (missing skipped: {missing})")
    return records


class COCO2017CaptionDataset(Dataset):
    '''Returns float32 images in [-1, 1] paired with a random human caption from COCO.'''
    def __init__(self, records, img_res: int = 256, random_caption: bool = True):
        self.records = records
        self.random_caption = random_caption
        self.transform = make_image_transform(img_res)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, caps = self.records[idx]
        with Image.open(path) as img:
            img = img.convert("RGB")
            pixel_values = self.transform(img)
        prompt = random.choice(caps) if self.random_caption else caps[0]
        return {"pixel_values": pixel_values, "prompts": prompt}


ACTIVE_DATASET_NAME = DATASET_MODE
if DATASET_MODE == "nature_50k":
    train_dataset, val_dataset = load_hf_nature_50k(
        NATURE_HF_REPO, val_size=NATURE_VAL_SIZE, seed=DATASET_SEED, cache_dir=NATURE_CACHE_DIR, img_res=256
    )
elif DATASET_MODE in ("coco2017_nature_50k", "coco2017_full"):
    nature_only = (DATASET_MODE == "coco2017_nature_50k")
    max_train = 50000 if nature_only else None
    ensure_coco_archive("annotations_trainval2017.zip", DATA_ROOT / "annotations")
    ensure_coco_archive("val2017.zip", DATA_ROOT / "val2017")
    if DOWNLOAD_TRAIN2017:
        ensure_coco_archive("train2017.zip", DATA_ROOT / "train2017")

    val_records = build_coco_records(
        DATA_ROOT / "val2017", DATA_ROOT / "annotations" / "captions_val2017.json",
        max_samples=(2000 if nature_only else None), nature_only=nature_only,
    )
    if DOWNLOAD_TRAIN2017:
        train_records = build_coco_records(
            DATA_ROOT / "train2017", DATA_ROOT / "annotations" / "captions_train2017.json",
            max_samples=max_train, nature_only=nature_only,
        )
    else:
        print("DOWNLOAD_TRAIN2017=False -> using val2017 as training set for dry run.")
        train_records = val_records

    train_dataset = COCO2017CaptionDataset(train_records, img_res=256, random_caption=True)
    val_dataset = COCO2017CaptionDataset(val_records, img_res=256, random_caption=False)
else:
    raise ValueError(f"Unsupported DATASET_MODE={DATASET_MODE!r}")

print(f"\nDataset Ready [{ACTIVE_DATASET_NAME}]: train={len(train_dataset):,} | val={len(val_dataset):,}")

# Visual sanity check on 4 samples
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    sample = train_dataset[i % len(train_dataset)]
    vis = ((sample["pixel_values"].permute(1, 2, 0).numpy() + 1.0) * 0.5).clip(0, 1)
    ax.imshow(vis)
    ax.set_title(sample["prompts"][:45] + ("..." if len(sample["prompts"]) > 45 else ""), fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Dataset VAE Whitening Stats & Compression Ratio Verification
Two critical checks before training:
1. **Latent Whitening (`raw_mean`, `raw_std`):** `Utils/configs.py` hardcodes LHQ-only `(0.566, -0.886, 1.47, -0.413)` and `(5.215, 5.469, 4.044, 3.988)`. Here we compute the active dataset's true latent channel statistics directly at `256×256` -> `32×32×4` so normalized latents satisfy mean $\approx 0.0$ and std $\approx \sigma_{\text{data}}$ (`0.8` by default).
2. **VAE Reconstruction & Architecture Compression Inspection:** Measures VAE round-trip PSNR/MSE on `val_dataset` and prints every tensor dimension / compression ratio across VAE, `Unet_expert`, `Vit_expert`, and `PooledCrossAttn`.


In [ ]:
# @title 3. Optional Drive VAE + CLIP Embedding Cache & Dataset Whitening Stats
import math
import shutil
import numpy as np
import torch
from tqdm.auto import tqdm
from Utils.VAE_CLIP import CLIP_EMBED, StabilityVAE
from Utils.embedding_cache import build_or_load_embedding_cache

SIGMA_DATA = 0.8  # Matches Utils/configs.py model_configs['sigma_data']
ENABLE_EMBEDDING_CACHE = True  # @param {type:"boolean"}
CACHE_CLIP_EMBEDDINGS = True   # @param {type:"boolean"}
CACHE_MAX_CAPTIONS_PER_IMAGE = 1  # @param {type:"integer"}  # 1 recommended for COCO-2017 (~14GB CLIP vs ~70GB for all 5); 0 = all captions
CACHE_PRELOAD_VAE_TO_RAM = True   # @param {type:"boolean"}  # Fast in-RAM reads when CACHE_CLIP_EMBEDDINGS=False
COPY_CACHE_TO_LOCAL_NVME = True   # @param {type:"boolean"}  # Mirror Drive cache to /content/emb_cache for high throughput
FORCE_REBUILD_CACHE = False       # @param {type:"boolean"}
WHITEN_SAMPLES = 2048             # @param {type:"integer"}  # Used only when ENABLE_EMBEDDING_CACHE=False
WHITEN_BATCH_SIZE = 64
CACHE_ENCODE_BATCH_SIZE = 64

_DS_TAG = globals().get("ACTIVE_DATASET_NAME", "dataset")
_ACTIVE_MODE = globals().get("DATASET_MODE", _DS_TAG)
_DRIVE_CACHE_ROOT = globals().get("DRIVE_BASE_DIR", Path("/content/drive/MyDrive/HDMOEM_colab")) / "embedding_cache" / _ACTIVE_MODE

if ENABLE_EMBEDDING_CACHE:
    cap_tag = f"cap{CACHE_MAX_CAPTIONS_PER_IMAGE}" if CACHE_CLIP_EMBEDDINGS else "vae_only"
    drive_train_cache = _DRIVE_CACHE_ROOT / f"train_{cap_tag}"
    drive_val_cache = _DRIVE_CACHE_ROOT / f"val_{cap_tag}"
    local_train_cache = Path("/content/emb_cache") / _ACTIVE_MODE / f"train_{cap_tag}" if COPY_CACHE_TO_LOCAL_NVME else None
    local_val_cache = Path("/content/emb_cache") / _ACTIVE_MODE / f"val_{cap_tag}" if COPY_CACHE_TO_LOCAL_NVME else None

    if USE_GOOGLE_DRIVE:
        usage = shutil.disk_usage(str(DRIVE_BASE_DIR))
        free_gb = usage.free / (1024 ** 3)
        est_vae_gb = len(train_dataset) * 4 * 32 * 32 * 2 * 2 / (1024 ** 3)
        caps_est = max(1, CACHE_MAX_CAPTIONS_PER_IMAGE) if _ACTIVE_MODE.startswith("coco") and CACHE_MAX_CAPTIONS_PER_IMAGE > 0 else (5 if _ACTIVE_MODE.startswith("coco") else 1)
        est_clip_gb = (len(train_dataset) * caps_est * 77 * 768 * 2 / (1024 ** 3)) if CACHE_CLIP_EMBEDDINGS else 0.0
        print(f"Drive free space: {free_gb:.1f} GB | Est. train cache size -> VAE: {est_vae_gb:.2f} GB, CLIP: {est_clip_gb:.2f} GB")

    vae_probe = StabilityVAE(batch_size=CACHE_ENCODE_BATCH_SIZE)
    vae_probe.init("cuda")
    if CACHE_CLIP_EMBEDDINGS and ("clip_embedder" not in globals() or clip_embedder is None):
        clip_embedder = CLIP_EMBED(device="cuda")
        clip_embedder.init()
    _clip_for_cache = clip_embedder if CACHE_CLIP_EMBEDDINGS else None

    train_cached_dataset = build_or_load_embedding_cache(
        cache_dir=drive_train_cache,
        dataset=train_dataset,
        vae=vae_probe,
        clip=_clip_for_cache,
        dataset_name=_ACTIVE_MODE,
        split="train",
        img_res=256,
        cache_clip=CACHE_CLIP_EMBEDDINGS,
        max_captions_per_image=CACHE_MAX_CAPTIONS_PER_IMAGE if CACHE_CLIP_EMBEDDINGS else 0,
        encode_batch_size=CACHE_ENCODE_BATCH_SIZE,
        clip_batch_size=256,
        num_workers=2,
        device="cuda",
        use_amp=True,
        overwrite=FORCE_REBUILD_CACHE,
        local_cache_dir=local_train_cache,
        preload_vae_to_ram=CACHE_PRELOAD_VAE_TO_RAM,
        random_caption=True,
    )
    val_cached_dataset = build_or_load_embedding_cache(
        cache_dir=drive_val_cache,
        dataset=val_dataset,
        vae=vae_probe,
        clip=_clip_for_cache,
        dataset_name=_ACTIVE_MODE,
        split="val",
        img_res=256,
        cache_clip=CACHE_CLIP_EMBEDDINGS,
        max_captions_per_image=1,
        encode_batch_size=CACHE_ENCODE_BATCH_SIZE,
        clip_batch_size=256,
        num_workers=2,
        device="cuda",
        use_amp=True,
        overwrite=FORCE_REBUILD_CACHE,
        local_cache_dir=local_val_cache,
        preload_vae_to_ram=CACHE_PRELOAD_VAE_TO_RAM,
        random_caption=False,
    )

    COCO_RAW_MEAN = train_cached_dataset.raw_mean
    COCO_RAW_STD = train_cached_dataset.raw_std
    active_train_dataset = train_cached_dataset
    active_val_dataset = val_cached_dataset
    print(f"\nExact analytical raw stats across {len(train_cached_dataset):,} cached images [{_DS_TAG}]:")
    print(f"  COCO_RAW_MEAN = {COCO_RAW_MEAN}")
    print(f"  COCO_RAW_STD  = {COCO_RAW_STD}")
else:
    whiten_loader = DataLoader(
        train_dataset, batch_size=WHITEN_BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True, drop_last=False
    )
    vae_probe = StabilityVAE(batch_size=WHITEN_BATCH_SIZE)
    vae_probe.init("cuda")

    ch_sum = torch.zeros(4, dtype=torch.float64, device="cuda")
    ch_sq_sum = torch.zeros(4, dtype=torch.float64, device="cuda")
    n_elems = 0
    seen = 0
    with torch.no_grad():
        for batch in tqdm(whiten_loader, desc=f"Computing {_DS_TAG} VAE raw stats"):
            imgs_m1_1 = batch["pixel_values"].to("cuda", non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                mean, std = vae_probe._run_vae_encoder_on_batch(imgs_m1_1)
                z_raw = mean.float() + torch.randn_like(mean.float()) * std.float()
            ch_sum += z_raw.sum(dim=(0, 2, 3), dtype=torch.float64)
            ch_sq_sum += (z_raw ** 2).sum(dim=(0, 2, 3), dtype=torch.float64)
            n_elems += z_raw.shape[0] * z_raw.shape[2] * z_raw.shape[3]
            seen += z_raw.shape[0]
            if seen >= WHITEN_SAMPLES:
                break

    raw_mean_tensor = (ch_sum / n_elems).cpu()
    raw_var_tensor = (ch_sq_sum / n_elems).cpu() - raw_mean_tensor ** 2
    raw_std_tensor = torch.sqrt(torch.clamp(raw_var_tensor, min=1e-12))
    COCO_RAW_MEAN = tuple(round(float(v), 5) for v in raw_mean_tensor.tolist())
    COCO_RAW_STD = tuple(round(float(v), 5) for v in raw_std_tensor.tolist())
    active_train_dataset = train_dataset
    active_val_dataset = val_dataset
    print(f"\nMeasured on {seen} images [{_DS_TAG}] (256x256 -> 32x32x4):")
    print(f"  COCO_RAW_MEAN = {COCO_RAW_MEAN}")
    print(f"  COCO_RAW_STD  = {COCO_RAW_STD}")

vae_verified = StabilityVAE(
    batch_size=WHITEN_BATCH_SIZE,
    raw_mean=COCO_RAW_MEAN,
    raw_std=COCO_RAW_STD,
    final_mean=0.0,
    final_std=SIGMA_DATA,
)
vae_verified.init("cuda")

val_check_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
val_batch = next(iter(val_check_loader))["pixel_values"].to("cuda")
with torch.no_grad():
    imgs_0_255 = ((val_batch + 1.0) * 127.5).clamp(0, 255)
    z_norm = vae_verified.encode(imgs_0_255)
    recon_0_255 = vae_verified.decode(z_norm).float()

mse_pix = torch.mean(((imgs_0_255 - recon_0_255) / 255.0) ** 2).item()
psnr_db = -10.0 * math.log10(max(mse_pix, 1e-12))
print("\nValidation Batch Verification:")
print(f"  Normalized latent shape : {tuple(z_norm.shape)}")
print(f"  Channel means (target 0): {[round(float(x), 4) for x in z_norm.mean(dim=(0, 2, 3)).tolist()]}")
print(f"  Channel stds  (target {SIGMA_DATA}): {[round(float(x), 4) for x in z_norm.std(dim=(0, 2, 3)).tolist()]}")
print(f"  Global mean: {z_norm.mean().item():.5f} | Global std: {z_norm.std().item():.5f}")
print(f"  VAE 48:1 Round-trip MSE: {mse_pix:.6f} | PSNR: {psnr_db:.2f} dB")

del vae_probe
torch.cuda.empty_cache()


## 4. Colab A100 Nature-50k Config, `ThermodynamicProbe`, & Repo Engine Setup (`Utils.training.training_HDMOE`)
All core fixes are now integrated directly into the repository modules — **no in-notebook monkey patching or duplicated training loops**:
* **`WSDScheduler` (`Utils/utils.py`) & Balanced Component Groups (`Utils.training.build_optimizer_param_groups`):** Linear warmup (`5%` of steps, clamped `20..400`) → flat `1.0×` plateau through `85%` of `TOTAL_STEPS` → cosine decay over the final `15%` to `0.10×`. Partitions `gate_params` (`2e-3`), `Unet_experts` / `VIT_experts` / `output_proj_*` (`1.2e-3`), `router` / `pooled_cross_attn` (`1.0e-3`), `default` (`8e-4`), and `blend_residual` (`3e-4`).
* **Native Architecture & BF16 Safety (`models/model_config_v3.py`, `models/model_internals.py`):** Native `3×3` `output_proj_vit`, `[1, 2]` hybrid patch ordering (`p=1` at low $\sigma$), BF16-safe `mp_sum`/`mp_cat`, and configurable `max_blend_shift = 0.40` + `blend_anchor_weight = 0.02` + `cfg_drop_prob = 0.15`.
* **Native Repo Training Engine (`Utils.training.training_HDMOE`) & Diagnostics (`graphs/logger2.py`, `graphs/plotter2.py`):** Accepts HF `nature_50k` `Dataset` (`train_dataset`, `val_dataset`), reuses pre-calibrated `vae_verified` / `clip_embedder`, handles BF16 autocast, micro-batch gradient accumulation, optional activation checkpointing, deterministic `Val Probe MSE`, checkpoint rotation, and Drive sync callbacks.

In [ ]:
# @title 4. Define Colab A100 Nature-50k Configs, `ThermodynamicProbe`, & Repo Training Wrapper
import copy
import math
import os
import shutil
import time
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn

from models.model_config_v3 import preconditioned_HDMOEM_v3
from Utils.configs import (
    loss_configs as base_loss_configs,
    mask_configs as base_mask_configs,
    model_configs as base_model_configs,
    optim_configs as base_optim_configs,
    zeta_configs as base_zeta_configs,
)
from Utils.loss import EDM_LOSS_NOD
from Utils.training import (
    apply_activation_checkpointing,
    build_optimizer_param_groups,
    training_HDMOE,
)
from Utils.utils import EMA, MaskGenerator, WSDScheduler, ZetaScheduler
from Utils.VAE_CLIP import CLIP_EMBED, StabilityVAE

# Enable TensorCore TF32 acceleration on A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")


ARCH_PROFILES_300M = {
    # Optimal <=300M: 293.60M total | UNet=128.95M (43.9%) | ViT=163.58M (55.7%) | Shared=1.08M
    "balanced_300m_p2": {
        "internal_channels": 128,
        "Unet_model_channels": 96,
        "Unet_channel_mult": [1, 2, 4],
        "Unet_num_blocks": 2,
        "Unet_kernel_sizes": [(3, 3), (5, 5)],
        "VIT_num_blocks": 10,
        "VIT_patch_sizes": [2, 2],
        "VIT_num_heads": 8,
        "VIT_emb_size": 512,
        "VIT_num_groups": 4,
    },
    # Hybrid patch <=300M: 275.38M total | UNet=128.95M (46.8%) | ViT=145.36M (52.8%, e0 p=1 fine & e1 p=2 mid)
    "hybrid_300m_p12": {
        "internal_channels": 128,
        "Unet_model_channels": 96,
        "Unet_channel_mult": [1, 2, 4],
        "Unet_num_blocks": 2,
        "Unet_kernel_sizes": [(3, 3), (5, 5)],
        "VIT_num_blocks": 6,
        "VIT_patch_sizes": [1, 2],
        "VIT_num_heads": 8,
        "VIT_emb_size": 512,
        "VIT_num_groups": 4,
    },
    # Pure patch=1 <=300M: 285.74M total | UNet=128.95M (45.1%) | ViT=155.71M (54.5%)
    "fullres_300m_p1": {
        "internal_channels": 128,
        "Unet_model_channels": 96,
        "Unet_channel_mult": [1, 2, 4],
        "Unet_num_blocks": 2,
        "Unet_kernel_sizes": [(3, 3), (5, 5)],
        "VIT_num_blocks": 6,
        "VIT_patch_sizes": [1, 1],
        "VIT_num_heads": 8,
        "VIT_emb_size": 384,
        "VIT_num_groups": 4,
    },
}


def build_colab_configs(
    total_steps: int = 8000,
    micro_batch_size: int = 16,
    grad_accum_steps: int = 8,
    arch_profile: str = "balanced_300m_p2",
    vit_patch_size: Optional[int] = None,
    use_grad_checkpointing: bool = False,
    pool_size: int = 16,
    raw_mean=None,
    raw_std=None,
    sigma_data: float = SIGMA_DATA,
    dataset_size: Optional[int] = None,
    save_interval: int = 500,
    log_interval: int = 10,
    max_blend_shift: float = 0.40,
    blend_anchor_weight: float = 0.02,
    keep_last_ckpts: int = 2,
    cfg_drop_prob: float = 0.15,
):
    if arch_profile == "hybrid_300m_p21":
        print("[Config] Mapping legacy 'hybrid_300m_p21' -> 'hybrid_300m_p12' (VIT_patch_sizes=[1, 2]).")
        arch_profile = "hybrid_300m_p12"
    if arch_profile not in ARCH_PROFILES_300M:
        raise ValueError(f"Unknown arch_profile '{arch_profile}'. Choose from {list(ARCH_PROFILES_300M.keys())}")
    prof = copy.deepcopy(ARCH_PROFILES_300M[arch_profile])
    if vit_patch_size is not None:
        prof["VIT_patch_sizes"] = [int(vit_patch_size), int(vit_patch_size)]

    effective_batch = micro_batch_size * grad_accum_steps
    raw_mean = COCO_RAW_MEAN if raw_mean is None else raw_mean
    raw_std = COCO_RAW_STD if raw_std is None else raw_std
    ds_size = dataset_size if dataset_size is not None else len(globals().get('active_train_dataset', train_dataset))
    mse_pass_thresh = 0.20 * (float(sigma_data) / 0.8) ** 2

    m_cfg = copy.deepcopy(base_model_configs)
    m_cfg.update({
        "arch_profile": arch_profile,
        "run_name": RUN_NAME,
        "final_ckpt_name": "final_model.pt",
        "save_dir": str(Path("/content/checkpoints") / RUN_NAME),
        "save_dir_stats": str(Path("/content/logs") / RUN_NAME),
        "data_dir": str(DATA_ROOT),
        "device": "cuda",
        "use_amp": True,
        "num_workers": 4,
        "dataset_size": ds_size,
        "img_channels": 4,
        "internal_channels": prof["internal_channels"],
        "data_img_res": 256,
        "img_resolution": 32,
        "text_emb_dim": 768,  # Matches openai/clip-vit-large-patch14
        "Unet_model_channels": prof["Unet_model_channels"],
        "Unet_channel_mult": prof["Unet_channel_mult"],
        "Unet_num_blocks": prof["Unet_num_blocks"],
        "Unet_kernel_sizes": prof["Unet_kernel_sizes"],
        "VIT_num_blocks": prof["VIT_num_blocks"],
        "VIT_patch_sizes": prof["VIT_patch_sizes"],
        "VIT_num_heads": prof["VIT_num_heads"],
        "VIT_emb_size": prof["VIT_emb_size"],
        "VIT_num_groups": prof["VIT_num_groups"],
        "cfg_drop_prob": float(cfg_drop_prob),
        "pool_size": pool_size,
        "sigma_data": sigma_data,
        "mean_data": 0.0,
        "raw_mean": tuple(raw_mean),
        "raw_std": tuple(raw_std),
        "batch_size": effective_batch,
        "micro_batch_size": micro_batch_size,
        "grad_accum_steps": grad_accum_steps,
        "use_grad_checkpointing": use_grad_checkpointing,
        "total_steps": total_steps,
        "save_interval": save_interval,
        "log_interval": log_interval,
        "keep_last_ckpts": keep_last_ckpts,
        "max_blend_shift": float(max_blend_shift),
        "blend_anchor_weight": float(blend_anchor_weight),
        "mse_pass_threshold": float(mse_pass_thresh),
        "ema_beta": 0.999,
        "ema_warmup_power": 2.0 / 3.0,
        "ema_min_warmup_step": max(150, int(0.15 * total_steps)),
    })

    warmup_steps = max(20, min(400, int(0.05 * total_steps)))
    rampup_mimg = max(0.005, (warmup_steps * effective_batch) / 1e6)

    o_cfg = copy.deepcopy(base_optim_configs)
    o_cfg.update({
        "lr_schedule": "wsd",
        "lr_unet": 1.2e-3,
        "lr_vit": 1.2e-3,
        "lr_router": 1.0e-3,
        "lr_out_proj_vit": 1.2e-3,
        "lr_out_proj_unet": 1.2e-3,
        "lr_pooled_attn": 1.0e-3,
        "lr_blend_residual": 3.0e-4,
        "lr_gates": 2.0e-3,
        "LR_DEFAULT": 8.0e-4,
        "grad_clip": 5.0,
        "warmup_steps": warmup_steps,
        "stable_until_frac": 0.85,
        "min_lr_mult": 0.10,
        "rampup_Mimg": rampup_mimg,
        "ref_batches": 0.0,
    })

    l_cfg = copy.deepcopy(base_loss_configs)
    l_cfg.update({
        "laplacian_std": 0.5,
        "alpha_blend": 0.8,
        "alpha_unet": 0.1,
        "alpha_vit": 0.1,
    })
    mk_cfg = copy.deepcopy(base_mask_configs)
    mk_cfg["total_steps"] = total_steps

    z_cfg = copy.deepcopy(base_zeta_configs)
    z_cfg["max_zeta"] = 1.0
    z_cfg["warmup_ratio"] = 0.10
    z_cfg["total_schedule_steps"] = max(20, int(0.30 * total_steps))

    return m_cfg, o_cfg, l_cfg, mk_cfg, z_cfg


class ThermodynamicProbe:
    """Monitors internal variance stability across HDMOEM v3 layers at fixed sigmas."""
    def __init__(self, model: nn.Module):
        self.model = model
        self.stats = {}
        self.hooks = []

    def _hook(self, name: str):
        def fn(mod, inp, out):
            t = out[0] if isinstance(out, (tuple, list)) else out
            if isinstance(t, torch.Tensor):
                tf = t.detach().float()
                self.stats[name] = (float(tf.mean().item()), float(tf.var(unbiased=False).item()))
        return fn

    def attach(self):
        net = self.model.net
        targets = {
            "input_proj": net.input_proj,
            "output_proj_unet": net.output_proj_unet,
            "output_proj_vit": net.output_proj_vit,
        }
        for i, ue in enumerate(net.Unet_experts):
            if "32x32_block0" in ue.decoders:
                targets[f"unet{i}.dec_res2"] = ue.decoders["32x32_block0"].conv_res2
        for i, ve in enumerate(net.VIT_experts):
            targets[f"vit{i}.b0"] = ve.diffit[0]
            targets[f"vit{i}.b_last"] = ve.diffit[-1]
        if net.pooled_cross_attn is not None:
            targets["pooled_cross_attn"] = net.pooled_cross_attn
        for name, mod in targets.items():
            self.hooks.append(mod.register_forward_hook(self._hook(name)))

    def detach(self):
        for h in self.hooks:
            h.remove()
        self.hooks.clear()

    @torch.no_grad()
    def run_check(self, test_sigmas=(0.05, 0.5, 2.5, 10.0, 60.0), res=32, device="cuda"):
        was_training = self.model.training
        self.model.eval()
        self.attach()
        try:
            text_emb = torch.randn(1, 77, 768, device=device)
            mask = torch.ones(1, self.model.net.num_experts, device=device)
            print("\n=== Thermodynamic Variance Probe ===")
            for s in test_sigmas:
                self.stats.clear()
                x = torch.randn(1, 4, res, res, device=device) * math.sqrt(s ** 2 + self.model.sigma_data ** 2)
                sig_t = torch.tensor([s], device=device)
                with torch.amp.autocast(torch.device(device).type, dtype=torch.bfloat16):
                    out = self.model(
                        x=x, sigma=sig_t, text_emb=text_emb,
                        Unet_router_mask=mask, Vit_router_mask=mask, zeta=0.0,
                    )
                den_var = float(out["denoised"].float().var(unbiased=False).item())
                u_gate = float(self.model.net.unet_proj_gate.item())
                v_gate = float(self.model.net.vit_proj_gate.item())
                summary = " | ".join([f"{k}: var={v[1]:.3f}" for k, v in self.stats.items()])
                print(
                    f"  sigma={s:5.2f} | w_vit={out['w_vit'].item():.3f} w_unet={out['w_unet'].item():.3f} "
                    f"| gates=(u:{u_gate:.3f},v:{v_gate:.3f}) | D_x var={den_var:.3f}\n    {summary}"
                )
        finally:
            self.detach()
            if was_training:
                self.model.train()


def sync_to_drive(local_dir: str, drive_subdir: Path, prune_old_ckpts: int = 0):
    if not os.path.exists(local_dir):
        return
    drive_subdir.mkdir(parents=True, exist_ok=True)
    for item in Path(local_dir).iterdir():
        if item.is_file():
            shutil.copy2(item, drive_subdir / item.name)
    if prune_old_ckpts > 0:
        remote_ckpts = sorted(
            [c for c in drive_subdir.glob("ckpt_*.pt") if c.stem.split("_")[-1].isdigit()],
            key=lambda p: int(p.stem.split("_")[-1]),
        )
        for old in remote_ckpts[:-prune_old_ckpts]:
            old.unlink()


def prepare_drive_resume(m_cfg: dict, sync_drive: bool, resume_mode: str = "latest"):
    save_dir = Path(m_cfg["save_dir"])
    log_dir = Path(m_cfg["save_dir_stats"])
    save_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)
    if not sync_drive or resume_mode != "latest":
        return
    local_ckpts = [c for c in save_dir.glob("ckpt_*.pt") if c.stem.split("_")[-1].isdigit()]
    drive_ckpts_dir = DRIVE_RUN_DIR / "checkpoints"
    if not local_ckpts and drive_ckpts_dir.exists():
        drive_ckpts = sorted(
            [c for c in drive_ckpts_dir.glob("ckpt_*.pt") if c.stem.split("_")[-1].isdigit()],
            key=lambda p: int(p.stem.split("_")[-1]),
        )
        if drive_ckpts:
            latest_drive = drive_ckpts[-1]
            shutil.copy2(latest_drive, save_dir / latest_drive.name)
            print(f"[Drive Resume] Restored {latest_drive.name} from Google Drive.")
            drive_logs_dir = DRIVE_RUN_DIR / "logs"
            if drive_logs_dir.exists():
                for old_log in drive_logs_dir.glob(f"{m_cfg['run_name']}_*.jsonl"):
                    shutil.copy2(old_log, log_dir / old_log.name)


def make_drive_sync_callback(sync_drive: bool, keep_last_ckpts: int):
    if not sync_drive:
        return None

    def _callback(save_dir: str, log_dir: str, tag: str):
        sync_to_drive(log_dir, DRIVE_RUN_DIR / "logs")
        sync_to_drive(save_dir, DRIVE_RUN_DIR / "checkpoints", prune_old_ckpts=keep_last_ckpts)

    return _callback


print("[Repo Engine Ready] Using Utils.training.training_HDMOE + WSDScheduler + native v3 fixes.")


## 5. Zero-Waste Preflight Smoke Test (30 Steps + Probe + Speed / Credit Estimate)
Run this short 30-step check **before** committing to the full training run. It checks:
1. Exact peak A100 VRAM usage under `MICRO_BATCH_SIZE` and BF16 autocast.
2. Seconds per optimizer step & projected runtime for `TOTAL_STEPS` so you can budget your 200 Colab credits accurately.
3. Initial `ThermodynamicProbe` variance across $\sigma \in [0.05, 60.0]$ (should remain near $\sim 1.0$, avoiding the pre-fix `1535.83` / `79.45` variances).


In [ ]:
# @title 5. Run 30-Step Preflight Smoke Check & Throughput Estimator (`Utils.training.training_HDMOE`)
ARCH_PROFILE = "balanced_300m_p2"  # @param ["balanced_300m_p2", "hybrid_300m_p12", "fullres_300m_p1"]
SMOKE_MICRO_BATCH = 16             # @param {type:"integer"}  # 16 fits easily on p=[2,2]; use 4 (or enable checkpointing) for p=1
SMOKE_GRAD_ACCUM = 4
USE_GRAD_CHECKPOINTING = False     # @param {type:"boolean"}  # Recommended True only if using hybrid_300m_p12 or fullres_300m_p1 with larger micro-batch

smoke_m, smoke_o, smoke_l, smoke_mk, smoke_z = build_colab_configs(
    total_steps=30,
    micro_batch_size=SMOKE_MICRO_BATCH,
    grad_accum_steps=SMOKE_GRAD_ACCUM,
    arch_profile=ARCH_PROFILE,
    use_grad_checkpointing=USE_GRAD_CHECKPOINTING,
    pool_size=16,
    save_interval=999,
    log_interval=5,
)
smoke_m["save_dir"] = "/content/smoke_checkpoints"
smoke_m["save_dir_stats"] = "/content/smoke_logs"
smoke_m["run_name"] = f"{RUN_NAME}_smoke"

train_ds_for_smoke = globals().get("active_train_dataset", train_dataset)
_need_clip_smoke = not getattr(train_ds_for_smoke, "has_clip_cache", False)
if _need_clip_smoke and ("clip_embedder" not in globals() or clip_embedder is None):
    clip_embedder = CLIP_EMBED(device="cuda")
    clip_embedder.init()
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()

smoke_model, smoke_ema, _, _ = training_HDMOE(
    smoke_m,
    smoke_o,
    smoke_l,
    smoke_mk,
    smoke_z,
    resume_path=None,
    dataset=train_ds_for_smoke,
    vae=vae_verified if not hasattr(train_ds_for_smoke, "shards") else None,
    clip=globals().get("clip_embedder") if _need_clip_smoke else None,
    keep_last_ckpts=2,
)

dt_smoke = time.perf_counter() - t0
peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
sec_per_step = dt_smoke / 30.0
total_params = sum(p.numel() for p in smoke_model.parameters())

print(f"\n--- Preflight Summary (profile={ARCH_PROFILE}, patch={smoke_m['VIT_patch_sizes']}, micro_batch={SMOKE_MICRO_BATCH}, accum={SMOKE_GRAD_ACCUM}) ---")
print(f"Total Parameters   : {total_params:,} ({total_params/1e6:.2f}M)")
print(f"Peak VRAM Allocated: {peak_gb:.2f} GB")
print(f"Throughput         : {sec_per_step:.2f} sec / effective optimizer step ({(SMOKE_MICRO_BATCH * SMOKE_GRAD_ACCUM) / sec_per_step:.1f} imgs/sec)")
print(f"Projected Time 5k  : {(5000 * sec_per_step) / 3600:.2f} hrs")
print(f"Projected Time 8k  : {(8000 * sec_per_step) / 3600:.2f} hrs (~{(8000 * sec_per_step) / 3600 * 1.1:.2f} hrs w/ accum=8)")

ThermodynamicProbe(smoke_model).run_check(res=32, device="cuda")

del smoke_model, smoke_ema
torch.cuda.empty_cache()


## 5b. Architecture & Loss Overfitting Sanity Test (Small Subset Memorization)
Runs an isolated **memorization sanity test** on a small fixed subset (`OVERFIT_SUBSET_SIZE = 64` by default, configurable up to `512`) without touching main run checkpoints or Google Drive.

**Why run this?** It cleanly separates **undertraining / LR schedule** from **architecture, loss, or sampler bugs**:
* **Per-$\sigma$ MSE collapse:** Evaluates fixed-noise MSE across $\sigma \in \{0.1, 0.5, 2.5, 10.0, 40.0\}$ (both ViT fine-detail and U-Net composition regimes).
* **Ground-truth vs `EDM_Sampler` (`CFG=1.0` & `CFG=3.0`) + 1-step `D_x`:** Shows whether memorized prompts generate crisp, coherent images from pure Gaussian noise.
* **RAPSD & `blend_mode` toggle:** Checks whether mid-frequency RAPSD excess persists on memorized data and lets you directly compare `OVERFIT_BLEND_MODE = "spherical"` vs `"linear"`.

In [ ]:
# @title 5b. Run Small-Subset Overfitting Sanity Test (Architecture / Loss / Blend Check)
import gc
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torchvision.utils import make_grid, save_image

from models.model_config_v3 import preconditioned_HDMOEM_v3
from Utils.EDM_sampler import EDM_Sampler
from Utils.loss import EDM_LOSS_NOD
from Utils.training import apply_activation_checkpointing, build_optimizer_param_groups
from Utils.utils import EMA, MaskGenerator, WSDScheduler, ZetaScheduler, sample_sigma_hybrid
from Utils.VAE_CLIP import CLIP_EMBED

OVERFIT_ARCH_PROFILE = "balanced_300m_p2"  # @param ["balanced_300m_p2", "hybrid_300m_p12", "fullres_300m_p1"]
OVERFIT_BLEND_MODE = "spherical"           # @param ["spherical", "linear"]
OVERFIT_SUBSET_SIZE = 64                   # @param {type:"integer"}  # 32-64 runs in ~3 min; 512 needs ~4000+ steps
OVERFIT_TOTAL_STEPS = 2000                 # @param {type:"integer"}
OVERFIT_MICRO_BATCH = 16                   # @param {type:"integer"}
OVERFIT_GRAD_ACCUM = 2                     # @param {type:"integer"}  # Effective batch = 32 -> 1000 epochs on 64 imgs in 2000 steps
OVERFIT_CFG_DROP_PROB = 0.10               # @param {type:"number"}
OVERFIT_USE_GRAD_CKPT = False              # @param {type:"boolean"}
OVERFIT_EVAL_INTERVAL = 250                # @param {type:"integer"}
OVERFIT_SEED = 42                          # @param {type:"integer"}

if "train_dataset" not in globals() or "vae_verified" not in globals():
    raise RuntimeError("Run Sections 2 and 3 first to initialize `train_dataset` and `vae_verified`.")

device = "cuda" if torch.cuda.is_available() else "cpu"
if "clip_embedder" not in globals() or clip_embedder is None:
    clip_embedder = CLIP_EMBED(device=device)
clip_embedder.init()

# 1. Pre-encode fixed subset once into VRAM (skips VAE/CLIP/DataLoader overhead during the loop)
n_overfit = min(int(OVERFIT_SUBSET_SIZE), len(train_dataset))
micro_bs = min(int(OVERFIT_MICRO_BATCH), n_overfit)
grad_accum = max(1, int(OVERFIT_GRAD_ACCUM))
subset_indices = np.linspace(0, len(train_dataset) - 1, n_overfit, dtype=int).tolist()

raw_imgs = torch.stack([train_dataset[i]["pixel_values"] for i in subset_indices])
overfit_prompts = [train_dataset[i]["prompts"] for i in subset_indices]
print(f"[Overfit Test] Subset={n_overfit} images | Unique prompts={len(set(overfit_prompts))}/{n_overfit} | blend_mode={OVERFIT_BLEND_MODE}")
for idx in range(min(4, n_overfit)):
    print(f"  [{idx}] {overfit_prompts[idx][:80]}")

x0_chunks, emb_chunks = [], []
with torch.no_grad():
    for s_idx in range(0, n_overfit, 32):
        e_idx = min(n_overfit, s_idx + 32)
        px_0_255 = ((raw_imgs[s_idx:e_idx].to(device) + 1.0) * 127.5).clamp(0, 255)
        with torch.amp.autocast(torch.device(device).type, dtype=torch.bfloat16):
            x0_chunks.append(vae_verified.encode(px_0_255).float())
            emb_chunks.append(clip_embedder.encode_text(overfit_prompts[s_idx:e_idx]).float())
    overfit_x0 = torch.cat(x0_chunks, dim=0)
    overfit_text_emb = torch.cat(emb_chunks, dim=0)
    cached_null_emb = clip_embedder.encode_text([""]).float().detach()
    n_vis = min(4, n_overfit)
    gt_vae_vis = vae_verified.decode(overfit_x0[:n_vis]).float().cpu() / 255.0

del raw_imgs, x0_chunks, emb_chunks
torch.cuda.empty_cache()

# 2. Build model & components directly from repository modules
torch.manual_seed(OVERFIT_SEED)
ov_m, ov_o, ov_l, ov_mk, ov_z = build_colab_configs(
    total_steps=OVERFIT_TOTAL_STEPS,
    micro_batch_size=micro_bs,
    grad_accum_steps=grad_accum,
    arch_profile=OVERFIT_ARCH_PROFILE,
    use_grad_checkpointing=OVERFIT_USE_GRAD_CKPT,
    pool_size=16,
    dataset_size=n_overfit,
    save_interval=OVERFIT_TOTAL_STEPS + 1,
    log_interval=10,
    cfg_drop_prob=OVERFIT_CFG_DROP_PROB,
)
ov_m["blend_mode"] = OVERFIT_BLEND_MODE
ov_m["ema_beta"] = min(0.999, 0.995) if OVERFIT_TOTAL_STEPS <= 4000 else 0.999
ov_m["ema_min_warmup_step"] = min(150, max(50, int(0.10 * OVERFIT_TOTAL_STEPS)))

ov_model = preconditioned_HDMOEM_v3(
    IN_in_channels=ov_m["img_channels"],
    IN_img_resolution=ov_m["img_resolution"],
    internal_channels=ov_m["internal_channels"],
    time_emb_dim=ov_m["time_emb_dim"],
    text_emb_dim=ov_m["text_emb_dim"],
    num_experts=ov_m["num_experts"],
    top_k=ov_m["top_k"],
    Fourier_bandwidth=ov_m["fourier_bandwidth"],
    VIT_num_blocks=ov_m["VIT_num_blocks"],
    VIT_patch_sizes=ov_m["VIT_patch_sizes"],
    VIT_num_groups=ov_m["VIT_num_groups"],
    VIT_num_heads=ov_m["VIT_num_heads"],
    VIT_emb_size=ov_m["VIT_emb_size"],
    VIT_emb_balance=ov_m["VIT_emb_balances"],
    VIT_pos_emb_balance=ov_m["VIT_pos_emb_balances"],
    Unet_num_blocks=ov_m["Unet_num_blocks"],
    Unet_channel_mult=ov_m["Unet_channel_mult"],
    Unet_channel_mult_emb=ov_m["Unet_channel_mult_emb"],
    Unet_kernel_sizes=ov_m["Unet_kernel_sizes"],
    Unet_model_channels=ov_m["Unet_model_channels"],
    Unet_label_balance=ov_m["Unet_label_balance"],
    Unet_concat_balance=ov_m["Unet_concat_balances"],
    sigma_data=ov_m["sigma_data"],
    log_var_channels=ov_m["log_var_channels"],
    transition_sigma=ov_m["transition_sigma"],
    transition_sharpness=ov_m["transition_sharpness"],
    pool_size=ov_m["pool_size"],
    attn_pool_balance=ov_m["attn_pool_balance"],
    p_mean=ov_mk["p_mean"],
    p_std=ov_mk["p_std"],
    sigma_min=ov_m["sigma_min"],
    sigma_max=ov_m["sigma_max"],
    alpha_ext=ov_m["extreme_prob"],
    max_blend_shift=ov_m["max_blend_shift"],
    blend_mode=ov_m["blend_mode"],
).to(device)

if OVERFIT_USE_GRAD_CKPT:
    apply_activation_checkpointing(ov_model)

ov_groups, _ = build_optimizer_param_groups(ov_model, ov_o)
ov_opt = torch.optim.Adam(ov_groups, betas=ov_o["betas"], weight_decay=ov_o["weight_decay"], eps=ov_o["eps"])
ov_sched = WSDScheduler(
    optimizer=ov_opt,
    total_steps=OVERFIT_TOTAL_STEPS,
    batch_size=ov_m["batch_size"],
    warmup_steps=ov_o["warmup_steps"],
    stable_until_frac=ov_o["stable_until_frac"],
    min_lr_mult=ov_o["min_lr_mult"],
    dataset_size=n_overfit,
)
ov_ema = EMA(ov_model, beta=ov_m["ema_beta"], warmup_power=ov_m["ema_warmup_power"], min_warmup_step=ov_m["ema_min_warmup_step"])
ov_zeta = ZetaScheduler(total_steps=ov_z["total_schedule_steps"], max_zeta=ov_z["max_zeta"], min_zeta=ov_z["min_zeta"], strategy=ov_z["strategy"], warmup_ratio=ov_z["warmup_ratio"])
ov_u_gen = MaskGenerator(
    expert_attributes=ov_mk["unet_attr"], p_mean=ov_mk["p_mean"], p_std=ov_mk["p_std"],
    total_steps=OVERFIT_TOTAL_STEPS, min_active=ov_mk["min_active"], step_size=ov_mk["step_size"],
    min_bandwidth=ov_mk["min_BW_u"], bandwidth=ov_mk["max_BW_u"], strat_band=ov_mk["strat_band"],
    noise_range=ov_mk["unet_noise_range"], sigma_max=ov_m["sigma_max"], sigma_min=ov_m["sigma_min"], alpha_ext=ov_m["extreme_prob"],
)
ov_v_gen = MaskGenerator(
    expert_attributes=ov_mk["vit_attr"], p_mean=ov_mk["p_mean"], p_std=ov_mk["p_std"],
    total_steps=OVERFIT_TOTAL_STEPS, min_active=ov_mk["min_active"], step_size=ov_mk["step_size"],
    min_bandwidth=ov_mk["min_BW_v"], bandwidth=ov_mk["max_BW_v"], strat_band=ov_mk["strat_band"],
    noise_range=ov_mk["vit_noise_range"], sigma_max=ov_m["sigma_max"], sigma_min=ov_m["sigma_min"], alpha_ext=ov_m["extreme_prob"],
)
ov_crit = EDM_LOSS_NOD(
    expert_centers_vit=ov_v_gen.expert_centers, expert_centers_Unet=ov_u_gen.expert_centers,
    num_experts=ov_m["num_experts"], sigma_data=ov_m["sigma_data"], Unet_bal=ov_l["unet_bal"], vit_bal=ov_l["vit_bal"],
    z_bal=ov_l["z_bal"], prior_bal=ov_l["prior_bal"], p_mean=ov_mk["p_mean"], p_std=ov_mk["p_std"],
    sharpness=ov_l["sharpness"], transition_sigma=ov_m["transition_sigma"], overlap_width=ov_m["transition_sharpness"],
    alpha_vit=ov_l["alpha_vit"], alpha_unet=ov_l["alpha_unet"], alpha_blend=ov_l["alpha_blend"],
    lap_kernel=ov_l["laplacian_kernel"], lap_sigma=ov_l["laplacian_std"], sigma_min=ov_m["sigma_min"],
    sigma_max=ov_m["sigma_max"], alpha_ext=ov_m["extreme_prob"],
    vit_noise_range=ov_mk["vit_noise_range"], unet_noise_range=ov_mk["unet_noise_range"],
)

EVAL_SIGMAS = (0.1, 0.5, 2.5, 10.0, 40.0)
n_probe = min(32, n_overfit)
probe_gen = torch.Generator(device="cpu").manual_seed(1234)
probe_eps = torch.randn(overfit_x0[:n_probe].shape, generator=probe_gen, dtype=torch.float32).to(device)


@torch.no_grad()
def _eval_overfit_bins(eval_m):
    was_train = eval_m.training
    eval_m.eval()
    mses = {}
    try:
        for s_val in EVAL_SIGMAS:
            sig_t = torch.full((n_probe, 1, 1, 1), float(s_val), device=device)
            noisy = overfit_x0[:n_probe] + probe_eps * sig_t
            u_m = ov_u_gen(sigma=sig_t, step=OVERFIT_TOTAL_STEPS)
            v_m = ov_v_gen(sigma=sig_t, step=OVERFIT_TOTAL_STEPS)
            with torch.amp.autocast(torch.device(device).type, dtype=torch.bfloat16):
                out = eval_m(
                    x=noisy, sigma=sig_t, text_emb=overfit_text_emb[:n_probe],
                    Unet_router_mask=u_m, Vit_router_mask=v_m, zeta=0.0,
                )
            mses[s_val] = float(torch.mean((out["denoised"].float() - overfit_x0[:n_probe]) ** 2).item())
    finally:
        if was_train:
            eval_m.train()
    return mses


def _overfit_radial_spectrum(tensor_4d: torch.Tensor):
    x = tensor_4d.float() - tensor_4d.float().mean(dim=(-2, -1), keepdim=True)
    fft = torch.fft.fft2(x, norm="ortho")
    fft_shift = torch.fft.fftshift(fft, dim=(-2, -1))
    power = (fft_shift.real ** 2 + fft_shift.imag ** 2).mean(dim=(0, 1)).cpu().numpy()
    h, w = power.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.indices((h, w))
    r = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2).astype(int)
    tbin = np.bincount(r.ravel(), power.ravel())
    nr = np.bincount(r.ravel())
    return (tbin / np.maximum(nr, 1))[: h // 2]


_perm, _perm_pos = None, 0


def _next_overfit_batch(bs: int):
    global _perm, _perm_pos
    if _perm is None or _perm_pos + bs > n_overfit:
        _perm = torch.randperm(n_overfit, device=device)
        _perm_pos = 0
    idx = _perm[_perm_pos : _perm_pos + bs]
    _perm_pos += bs
    return overfit_x0[idx], overfit_text_emb[idx]


# 3. Overfitting Training Loop
hist = {"step": [], "loss": [], "pure": [], "vit": [], "unet": [], "mse": []}
bin_hist = {"step": [], **{s: [] for s in EVAL_SIGMAS}}
init_bins = _eval_overfit_bins(ov_model)
bin_hist["step"].append(0)
for s_val in EVAL_SIGMAS:
    bin_hist[s_val].append(init_bins[s_val])
print("[Step     0] Init bin MSE: " + " | ".join(f"s={s}: {init_bins[s]:.4f}" for s in EVAL_SIGMAS))

blend_anchor_w = float(ov_m.get("blend_anchor_weight", 0.02))
t_start = time.perf_counter()
ov_model.train()

for step in range(OVERFIT_TOTAL_STEPS):
    ov_opt.zero_grad(set_to_none=True)
    cur_zeta = ov_zeta.get_zeta(step=step)
    acc = {"loss": 0.0, "pure_loss": 0.0, "loss_texture": 0.0, "loss_structure": 0.0, "denoising": 0.0}

    for _ in range(grad_accum):
        x0_b, txt_b = _next_overfit_batch(micro_bs)
        if OVERFIT_CFG_DROP_PROB > 0.0:
            drop_mask = (torch.rand(x0_b.shape[0], 1, 1, device=device) < OVERFIT_CFG_DROP_PROB)
            null_exp = cached_null_emb.to(dtype=txt_b.dtype).expand_as(txt_b)
            txt_b = torch.where(drop_mask, null_exp, txt_b)

        sigma = sample_sigma_hybrid(
            batch_size=x0_b.shape[0],
            sigma_max=ov_m["sigma_max"], sigma_min=ov_m["sigma_min"],
            p_mean=ov_mk["p_mean"], p_std=ov_mk["p_std"],
            extreme_prob=ov_m["extreme_prob"], device=device,
        )
        noisy_x = x0_b + torch.randn_like(x0_b) * sigma
        u_mask = ov_u_gen(sigma=sigma, step=step)
        v_mask = ov_v_gen(sigma=sigma, step=step)

        with torch.amp.autocast(torch.device(device).type, dtype=torch.bfloat16):
            out_m = ov_model(
                x=noisy_x, sigma=sigma, text_emb=txt_b,
                Unet_router_mask=u_mask, Vit_router_mask=v_mask,
                zeta=cur_zeta, return_log_var=True,
            )
            ld = ov_crit(x=x0_b, noisy_x=noisy_x, sigma=sigma, out_model=out_m)
            step_loss = ld["loss"]
            bres = getattr(ov_model.net, "_last_bounded_residual", None)
            if bres is not None and blend_anchor_w > 0.0:
                step_loss = step_loss + blend_anchor_w * bres.float().pow(2).mean()
            ov_model.net._last_bounded_residual = None
            (step_loss / grad_accum).backward()

        for k in acc:
            acc[k] += float(ld[k].detach().item()) / grad_accum

    torch.nn.utils.clip_grad_norm_(ov_model.parameters(), 5.0)
    ov_opt.step()
    ov_sched.step()
    ov_ema.update()

    if step % 20 == 0 or step == OVERFIT_TOTAL_STEPS - 1:
        hist["step"].append(step)
        hist["loss"].append(acc["loss"])
        hist["pure"].append(acc["pure_loss"])
        hist["vit"].append(acc["loss_texture"])
        hist["unet"].append(acc["loss_structure"])
        hist["mse"].append(acc["denoising"])

    if (step + 1) % OVERFIT_EVAL_INTERVAL == 0 or step == OVERFIT_TOTAL_STEPS - 1:
        b_mses = _eval_overfit_bins(ov_ema.ema_model)
        bin_hist["step"].append(step + 1)
        for s_val in EVAL_SIGMAS:
            bin_hist[s_val].append(b_mses[s_val])
        elapsed = time.perf_counter() - t_start
        print(
            f"Step {step+1:4d}/{OVERFIT_TOTAL_STEPS} ({elapsed:5.1f}s) | loss={acc['loss']:.4f} | "
            f"ViT={acc['loss_texture']:.4f} | UNet={acc['loss_structure']:.4f} | "
            + " | ".join(f"s={s}:{b_mses[s]:.4f}" for s in EVAL_SIGMAS)
        )

raw_final_bins = _eval_overfit_bins(ov_model)
ema_final_bins = {s: bin_hist[s][-1] for s in EVAL_SIGMAS}
print("\n[Final Bin MSE Comparison] EMA vs Raw:")
print("  EMA: " + " | ".join(f"s={s}: {ema_final_bins[s]:.5f}" for s in EVAL_SIGMAS))
print("  Raw: " + " | ".join(f"s={s}: {raw_final_bins[s]:.5f}" for s in EVAL_SIGMAS))

# 4. Pre-skip vs Post-skip Cosine Similarity + 1-Step Denoising + Full ODE Sampling
eval_ov = ov_ema.ema_model.eval()
sig_grid = torch.exp(torch.linspace(math.log(0.01), math.log(60.0), 16, device=device))
cos_post, cos_pre = [], []
sd = float(ov_m["sigma_data"])
with torch.no_grad():
    for sv in sig_grid:
        st = sv.view(1, 1, 1, 1).expand(n_vis, 1, 1, 1)
        nx = overfit_x0[:n_vis] + probe_eps[:n_vis] * st
        um = ov_u_gen(sigma=st, step=OVERFIT_TOTAL_STEPS)
        vm = ov_v_gen(sigma=st, step=OVERFIT_TOTAL_STEPS)
        with torch.amp.autocast(torch.device(device).type, dtype=torch.bfloat16):
            om = eval_ov(
                x=nx, sigma=st, text_emb=overfit_text_emb[:n_vis],
                Unet_router_mask=um, Vit_router_mask=vm, zeta=0.0,
            )
        dv = om["vit_denoised"].float().flatten(1)
        du = om["unet_denoised"].float().flatten(1)
        cos_post.append(float(F.cosine_similarity(dv, du, dim=1).mean().item()))

        c_skip = (sd ** 2) / (sv ** 2 + sd ** 2)
        c_out = sv * sd / torch.sqrt(sv ** 2 + sd ** 2)
        fv = ((om["vit_denoised"].float() - c_skip * nx.float()) / c_out.clamp(min=1e-8)).flatten(1)
        fu = ((om["unet_denoised"].float() - c_skip * nx.float()) / c_out.clamp(min=1e-8)).flatten(1)
        cos_pre.append(float(F.cosine_similarity(fv, fu, dim=1).mean().item()))

    s_mid = torch.full((n_vis, 1, 1, 1), 2.5, device=device)
    nx_mid = overfit_x0[:n_vis] + probe_eps[:n_vis] * s_mid
    with torch.amp.autocast(torch.device(device).type, dtype=torch.bfloat16):
        om_mid = eval_ov(
            x=nx_mid, sigma=s_mid, text_emb=overfit_text_emb[:n_vis],
            Unet_router_mask=ov_u_gen(sigma=s_mid, step=OVERFIT_TOTAL_STEPS),
            Vit_router_mask=ov_v_gen(sigma=s_mid, step=OVERFIT_TOTAL_STEPS),
            zeta=0.0,
        )
    onestep_vis = vae_verified.decode(om_mid["denoised"].float()).float().cpu() / 255.0

    gen_noise = torch.randn(n_vis, 4, 32, 32, device=device, generator=torch.Generator(device=device).manual_seed(OVERFIT_SEED))
    null_batch = cached_null_emb.expand(n_vis, -1, -1)

    smp_c1 = EDM_Sampler(eval_ov, num_solve_steps=32, guidance=1.0, unet_mask_gen=ov_u_gen, vit_mask_gen=ov_v_gen, mask_step=OVERFIT_TOTAL_STEPS)
    lats_c1 = smp_c1.sample(gen_noise.clone(), text_emb=overfit_text_emb[:n_vis], uncond_text_emb=null_batch)
    vis_c1 = vae_verified.decode(lats_c1.float()).float().cpu() / 255.0

    smp_c3 = EDM_Sampler(eval_ov, num_solve_steps=32, guidance=3.0, unet_mask_gen=ov_u_gen, vit_mask_gen=ov_v_gen, mask_step=OVERFIT_TOTAL_STEPS)
    lats_c3 = smp_c3.sample(gen_noise.clone(), text_emb=overfit_text_emb[:n_vis], uncond_text_emb=null_batch)
    vis_c3 = vae_verified.decode(lats_c3.float()).float().cpu() / 255.0

    spec_gt = _overfit_radial_spectrum(overfit_x0[:min(8, n_overfit)])
    spec_c1 = _overfit_radial_spectrum(lats_c1)
    spec_c3 = _overfit_radial_spectrum(lats_c3)

# 5. Plot Diagnostic Curves & Visual Comparison Grid
ov_dir = Path("/content/overfit_diagnostics")
ov_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].plot(hist["step"], hist["loss"], label="Total Loss", linewidth=2)
axes[0, 0].plot(hist["step"], hist["pure"], "--", label="Pure EDM Loss")
axes[0, 0].plot(hist["step"], hist["vit"], ":", label="ViT Branch (High-Freq)")
axes[0, 0].plot(hist["step"], hist["unet"], ":", label="UNet Branch (Low-Freq)")
axes[0, 0].set_title(f"(a) Overfit Loss ({n_overfit} imgs, blend={OVERFIT_BLEND_MODE})")
axes[0, 0].set_xlabel("Step"); axes[0, 0].legend(fontsize=8)

for s_val in EVAL_SIGMAS:
    axes[0, 1].semilogy(bin_hist["step"], np.maximum(bin_hist[s_val], 1e-6), marker="o", ms=3, label=f"sigma={s_val}")
axes[0, 1].set_title("(b) Per-Sigma Memorization MSE (log scale)")
axes[0, 1].set_xlabel("Step"); axes[0, 1].set_ylabel("MSE"); axes[0, 1].legend(fontsize=8)

s_np = sig_grid.cpu().numpy()
axes[1, 0].plot(s_np, cos_post, marker="o", ms=3, label="Post-skip D_x (ViT vs UNet)")
axes[1, 0].plot(s_np, cos_pre, marker="s", ms=3, label="Pre-skip raw F_x (ViT vs UNet)")
axes[1, 0].axvline(ov_m["transition_sigma"], color="gray", linestyle=":", label="transition_sigma")
axes[1, 0].set_xscale("log"); axes[1, 0].invert_xaxis()
axes[1, 0].set_title("(c) Branch Cosine Similarity (Pre- vs Post-Skip)")
axes[1, 0].set_xlabel("sigma"); axes[1, 0].legend(fontsize=8)

freqs = np.arange(len(spec_gt))
axes[1, 1].semilogy(freqs, spec_gt + 1e-12, linewidth=2, label="GT Overfit Latents")
axes[1, 1].semilogy(freqs, spec_c1 + 1e-12, "--", label="Generated (CFG=1.0)")
axes[1, 1].semilogy(freqs, spec_c3 + 1e-12, "-.", label="Generated (CFG=3.0)")
axes[1, 1].set_title("(d) Latent RAPSD on Memorized Subset")
axes[1, 1].set_xlabel("Spatial Frequency Radius"); axes[1, 1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(ov_dir / f"overfit_diagnostics_{OVERFIT_BLEND_MODE}.png", dpi=180)
plt.show()

grid = make_grid(torch.cat([gt_vae_vis, onestep_vis, vis_c1, vis_c3], dim=0), nrow=n_vis, padding=4)
save_image(grid, ov_dir / f"overfit_samples_{OVERFIT_BLEND_MODE}.png")
fig_img, ax_img = plt.subplots(figsize=(14, 11))
ax_img.imshow(grid.permute(1, 2, 0).numpy().clip(0, 1))
ax_img.set_title(
    f"Overfit Sanity Check ({n_overfit} imgs, {OVERFIT_TOTAL_STEPS} steps, blend={OVERFIT_BLEND_MODE})\n"
    "Rows (Top->Bottom): 1) GT VAE Recon  |  2) 1-Step D_x (sigma=2.5)  |  3) EDM_Sampler CFG=1.0  |  4) EDM_Sampler CFG=3.0",
    fontsize=11,
)
ax_img.axis("off")
plt.tight_layout()
plt.show()

# 6. Automated Diagnostic Verdict
mid_rapsd_ratio_c1 = float(np.mean(spec_c1[2:9] / np.maximum(spec_gt[2:9], 1e-12)))
mid_rapsd_ratio_c3 = float(np.mean(spec_c3[2:9] / np.maximum(spec_gt[2:9], 1e-12)))
pix_mse_c1 = float(torch.mean((vis_c1 - gt_vae_vis) ** 2).item())
pix_mse_c3 = float(torch.mean((vis_c3 - gt_vae_vis) ** 2).item())

print("\n=== Overfitting Test Summary ===")
print(f"  High-sigma MSE drop (s=10.0): {init_bins[10.0]:.4f} -> {ema_final_bins[10.0]:.5f}")
print(f"  Overlap    MSE drop (s=2.5) : {init_bins[2.5]:.4f} -> {ema_final_bins[2.5]:.5f}")
print(f"  Low-sigma  MSE drop (s=0.5) : {init_bins[0.5]:.4f} -> {ema_final_bins[0.5]:.5f}")
print(f"  Sample vs GT Pixel MSE      : CFG=1.0 -> {pix_mse_c1:.4f} | CFG=3.0 -> {pix_mse_c3:.4f}")
print(f"  Mid-freq RAPSD ratio (r=2..8): CFG=1.0 -> {mid_rapsd_ratio_c1:.2f}x | CFG=3.0 -> {mid_rapsd_ratio_c3:.2f}x")

if ema_final_bins[0.5] < 0.05 and ema_final_bins[10.0] < 0.10:
    print("  [PASS] Denoising loss & architecture can memorize both coarse layout (high sigma) and fine detail (low sigma).")
else:
    print("  [WARN] Memorization MSE stayed elevated — inspect per-sigma curves or increase OVERFIT_TOTAL_STEPS.")
if mid_rapsd_ratio_c1 > 1.5 and OVERFIT_BLEND_MODE == "spherical":
    print("  [NOTE] Mid-frequency RAPSD excess (>1.5x) observed at CFG=1.0; try re-running with OVERFIT_BLEND_MODE='linear' to isolate spherical_blend.")

del ov_model, ov_ema, ov_opt, ov_sched
del overfit_x0, overfit_text_emb, probe_eps
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# @title 6. Main Training Run on `nature_50k` (`Utils.training.training_HDMOE`, `<= 300M` params)
ARCH_PROFILE = "balanced_300m_p2"  # @param ["balanced_300m_p2", "hybrid_300m_p12", "fullres_300m_p1"]
TOTAL_STEPS = 8000                 # @param {type:"integer"}  # 8k * 128 = 1.024M samples (~21.3 epochs on 48k nature scenes)
MICRO_BATCH_SIZE = 16              # @param {type:"integer"}  # Match smoke test (bump to 32 with accum=4 if smoke Peak VRAM < 25GB)
GRAD_ACCUM_STEPS = 8               # @param {type:"integer"}  # 16 * 8 = 128 effective batch size
USE_GRAD_CHECKPOINTING = False     # @param {type:"boolean"}
SAVE_INTERVAL = 500                # @param {type:"integer"}
RESUME_TRAINING = "latest"         # @param ["latest", "none"]

main_m, main_o, main_l, main_mk, main_z = build_colab_configs(
    total_steps=TOTAL_STEPS,
    micro_batch_size=MICRO_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    arch_profile=ARCH_PROFILE,
    use_grad_checkpointing=USE_GRAD_CHECKPOINTING,
    pool_size=16,
    save_interval=SAVE_INTERVAL,
    log_interval=10,
)

prepare_drive_resume(main_m, sync_drive=USE_GOOGLE_DRIVE, resume_mode=RESUME_TRAINING)
drive_cb = make_drive_sync_callback(sync_drive=USE_GOOGLE_DRIVE, keep_last_ckpts=main_m["keep_last_ckpts"])

_train_ds = globals().get("active_train_dataset", train_dataset)
_val_ds = globals().get("active_val_dataset", val_dataset)
_using_vae_cache = hasattr(_train_ds, "shards") and hasattr(_val_ds, "shards")
_using_clip_cache = getattr(_train_ds, "has_clip_cache", False) and getattr(_val_ds, "has_clip_cache", False)
if not _using_clip_cache and ("clip_embedder" not in globals() or clip_embedder is None):
    clip_embedder = CLIP_EMBED(device="cuda")
    clip_embedder.init()

trained_model, trained_ema, active_cfg, training_logger = training_HDMOE(
    main_m,
    main_o,
    main_l,
    main_mk,
    main_z,
    resume_path=None if RESUME_TRAINING == "none" else "latest",
    dataset=_train_ds,
    val_dataset=_val_ds,
    vae=None if _using_vae_cache else vae_verified,
    clip=None if _using_clip_cache else clip_embedder,
    keep_last_ckpts=main_m["keep_last_ckpts"],
    post_save_callback=drive_cb,
)


In [ ]:
# @title 7. Plot All Training Diagnostics via `graphs.plotter2.Plotter`
from IPython.display import display
import shutil
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from pathlib import Path
from graphs.plotter2 import Plotter

run_name = globals().get("RUN_NAME", "hdmoem_v3_nature50k_256_v3")
if "active_cfg" in globals() and isinstance(active_cfg, dict) and "save_dir_stats" in active_cfg:
    log_dir = active_cfg["save_dir_stats"]
elif any((Path("/content/logs") / run_name).glob(f"{run_name}_*.jsonl")):
    log_dir = str(Path("/content/logs") / run_name)
else:
    log_dir = str(globals().get("DRIVE_RUN_DIR", Path("/content/drive/MyDrive/HDMOEM_colab") / run_name) / "logs")

ANALYSIS_OUT = str(Path("/content/analysis"))
plotter = Plotter(
    log_dir=log_dir,
    run_name=run_name,
    out_dir=ANALYSIS_OUT,
    smooth_sigma=5.0,
)
plotter.plot_all()

if globals().get("USE_GOOGLE_DRIVE", False):
    drive_analysis = globals().get("DRIVE_RUN_DIR", Path("/content/drive/MyDrive/HDMOEM_colab") / run_name) / "analysis"
    if "sync_to_drive" in globals():
        sync_to_drive(str(plotter.out_dir), drive_analysis)
    else:
        drive_analysis.mkdir(parents=True, exist_ok=True)
        for item in Path(plotter.out_dir).iterdir():
            if item.is_file():
                shutil.copy2(item, drive_analysis / item.name)

plot_files = sorted(Path(plotter.out_dir).glob("*.png"))
print(f"Generated {len(plot_files)} figures in {plotter.out_dir}")
for p in plot_files:
    print(f"\n--- {p.name} ---")
    img = mpimg.imread(str(p))
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    display(fig)
    plt.close(fig)


## 8. Post-Fix Trajectory & Internal-State Diagnostics + Masked Sampling Generation
Runs standalone from `final_model.pt` (even after a fresh runtime restart) OR immediately after training:
1. **Strict Sampling Router Masks:** Reconstructs `MaskGenerator` masks (`unet_mask_gen`, `vit_mask_gen`) at `mask_step = m_cfg["total_steps"]` and passes them into both `run_v3_trajectory_diagnostics` and `EDM_Sampler`, so inference obeys the exact quantile-partitioned expert routing used during training.
2. **Faithful Blend Anchor & ViT Kernel Reconstruction:** Applies `max_blend_shift` (`0.40` on fixed runs, `1.5` on legacy checkpoints) and reconstructs `output_proj_vit` with `vit_out_kernel` (`3×3` on v3 runs, `1×1` on legacy checkpoints) before loading `state_dict` so standalone evaluations match training behavior.
3. **Diagnostics & CFG Sweep:** Plots Lipschitz ODE velocity, ViT-vs-UNet cosine similarity, `w_vit`/`w_unet` handoff curves, RAPSD spectrum vs real validation latents, trajectory filmstrip (`guidance=3.0`), and CFG `1.0 / 3.0 / 5.0` sample grid.


In [ ]:
# @title 8. Load `final_model.pt`, Trajectory Probe (w/ Sampling Masks), RAPSD Check, & Sample Grid
import math
import shutil
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from torchvision.utils import make_grid, save_image

from models.model_config_v3 import preconditioned_HDMOEM_v3
from models.model_internals import MP_Conv
from Utils.configs import mask_configs as base_mask_configs
from Utils.EDM_sampler import EDM_Sampler
from Utils.utils import MaskGenerator
from Utils.VAE_CLIP import CLIP_EMBED, StabilityVAE

CKPT_PATH = ""          # @param {type:"string"}  # Leave blank to auto-locate final_model.pt, or paste full path
USE_EMA_WEIGHTS = True  # @param {type:"boolean"} # Recommended True: loads ema_model_state_dict when available


def build_eval_mask_generators(m_cfg: dict, mk_cfg: dict = None):
    cfg = {**base_mask_configs, **(mk_cfg or {})}
    total_steps = int(cfg.get("total_steps", m_cfg.get("total_steps", 8000)))
    t_sigma = float(m_cfg.get("transition_sigma", 2.5))
    s_min = float(m_cfg.get("sigma_min", 0.002))
    s_max = float(m_cfg.get("sigma_max", 80.0))
    a_ext = float(m_cfg.get("extreme_prob", 0.05))
    u_range = cfg.get("unet_noise_range", (t_sigma, s_max))
    v_range = cfg.get("vit_noise_range", (s_min, t_sigma))

    u_gen = MaskGenerator(
        expert_attributes=cfg["unet_attr"],
        p_mean=cfg["p_mean"],
        p_std=cfg["p_std"],
        total_steps=total_steps,
        min_active=cfg["min_active"],
        step_size=cfg["step_size"],
        min_bandwidth=cfg["min_BW_u"],
        bandwidth=cfg["max_BW_u"],
        strat_band=cfg["strat_band"],
        noise_range=u_range,
        sigma_max=s_max,
        sigma_min=s_min,
        alpha_ext=a_ext,
    )
    v_gen = MaskGenerator(
        expert_attributes=cfg["vit_attr"],
        p_mean=cfg["p_mean"],
        p_std=cfg["p_std"],
        total_steps=total_steps,
        min_active=cfg["min_active"],
        step_size=cfg["step_size"],
        min_bandwidth=cfg["min_BW_v"],
        bandwidth=cfg["max_BW_v"],
        strat_band=cfg["strat_band"],
        noise_range=v_range,
        sigma_max=s_max,
        sigma_min=s_min,
        alpha_ext=a_ext,
    )
    return u_gen, v_gen, total_steps


def resolve_final_checkpoint(ckpt_path: str = "") -> Path:
    if ckpt_path and ckpt_path.strip():
        p = Path(ckpt_path.strip())
        if not p.exists():
            raise FileNotFoundError(f"Specified checkpoint not found: {p}")
        return p

    run_name = globals().get("RUN_NAME", "hdmoem_v3_nature50k_256_v3")
    candidates = [
        Path("/content/checkpoints") / run_name / "final_model.pt",
        Path(globals().get("DRIVE_RUN_DIR", Path("/content/drive/MyDrive/HDMOEM_colab") / run_name)) / "checkpoints" / "final_model.pt",
        Path("/content/final_model.pt"),
        Path("final_model.pt"),
    ]
    if "active_cfg" in globals() and isinstance(active_cfg, dict) and "save_dir" in active_cfg:
        candidates.insert(0, Path(active_cfg["save_dir"]) / "final_model.pt")

    for cand in candidates:
        if cand.exists():
            return cand

    for root in (Path("/content"), Path("/content/drive/MyDrive/HDMOEM_colab")):
        if root.exists():
            found = sorted(root.rglob("final*model*.pt"))
            if found:
                return found[-1]

    raise FileNotFoundError(
        "Could not find final_model.pt automatically. Checked:\n  - "
        + "\n  - ".join(str(c) for c in candidates)
        + "\nSet CKPT_PATH to your checkpoint file path."
    )


def load_eval_model_from_checkpoint(ckpt_path: Path, use_ema: bool = True, device: str = "cuda"):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    saved_cfg = ckpt.get("config", {})
    m_cfg = saved_cfg.get("model_configs", saved_cfg)
    mk_cfg = saved_cfg.get("mask_config", {})

    model = preconditioned_HDMOEM_v3(
        IN_in_channels=m_cfg["img_channels"],
        IN_img_resolution=m_cfg["img_resolution"],
        internal_channels=m_cfg["internal_channels"],
        time_emb_dim=m_cfg["time_emb_dim"],
        text_emb_dim=m_cfg["text_emb_dim"],
        num_experts=m_cfg["num_experts"],
        top_k=m_cfg["top_k"],
        Fourier_bandwidth=m_cfg["fourier_bandwidth"],
        VIT_num_blocks=m_cfg["VIT_num_blocks"],
        VIT_patch_sizes=m_cfg["VIT_patch_sizes"],
        VIT_num_groups=m_cfg["VIT_num_groups"],
        VIT_num_heads=m_cfg["VIT_num_heads"],
        VIT_emb_size=m_cfg["VIT_emb_size"],
        VIT_emb_balance=m_cfg["VIT_emb_balances"],
        VIT_pos_emb_balance=m_cfg["VIT_pos_emb_balances"],
        Unet_num_blocks=m_cfg["Unet_num_blocks"],
        Unet_channel_mult=m_cfg["Unet_channel_mult"],
        Unet_channel_mult_emb=m_cfg["Unet_channel_mult_emb"],
        Unet_kernel_sizes=m_cfg["Unet_kernel_sizes"],
        Unet_model_channels=m_cfg["Unet_model_channels"],
        Unet_label_balance=m_cfg["Unet_label_balance"],
        Unet_concat_balance=m_cfg["Unet_concat_balances"],
        sigma_data=m_cfg["sigma_data"],
        log_var_channels=m_cfg["log_var_channels"],
        transition_sigma=m_cfg["transition_sigma"],
        transition_sharpness=m_cfg["transition_sharpness"],
        pool_size=m_cfg["pool_size"],
        attn_pool_balance=m_cfg["attn_pool_balance"],
        p_mean=mk_cfg.get("p_mean", base_mask_configs["p_mean"]),
        p_std=mk_cfg.get("p_std", base_mask_configs["p_std"]),
        sigma_min=m_cfg["sigma_min"],
        sigma_max=m_cfg["sigma_max"],
        alpha_ext=m_cfg["extreme_prob"],
        max_blend_shift=float(m_cfg.get("max_blend_shift", 1.5)),
        blend_mode=m_cfg.get("blend_mode", "spherical"),
    ).to(device)

    state_key = "ema_model_state_dict" if (use_ema and "ema_model_state_dict" in ckpt) else "model_state_dict"
    state_dict = ckpt[state_key]
    w_proj = state_dict.get("net.output_proj_vit.weight", state_dict.get("net.output_proj_vit.weights", None))
    if w_proj is not None and w_proj.ndim == 4:
        inferred_k = int(w_proj.shape[-1])
        cur_k = int(model.net.output_proj_vit.weight.shape[-1])
        if inferred_k != cur_k:
            model.net.output_proj_vit = MP_Conv(
                in_channels=m_cfg["internal_channels"] + 1,
                out_channels=m_cfg["img_channels"],
                kernel=(inferred_k, inferred_k),
            ).to(device)

    model.load_state_dict(state_dict, strict=True)
    model.requires_grad_(False).eval()

    u_gen, v_gen, mask_step = build_eval_mask_generators(m_cfg, mk_cfg)
    model._eval_unet_mask_gen = u_gen
    model._eval_vit_mask_gen = v_gen
    model._eval_mask_step = mask_step

    mse_val = ckpt.get("mse", float("nan"))
    mse_str = f"{mse_val:.5f}" if isinstance(mse_val, (int, float)) else str(mse_val)
    print(
        f"[Eval] Loaded '{state_key}' from {ckpt_path} "
        f"(step={ckpt.get('step', '?')}, mse={mse_str}, max_blend_shift={model.net._max_blend_shift:.2f}, mask_step={mask_step})"
    )
    return model, m_cfg


def compute_radial_spectrum(tensor_4d: torch.Tensor):
    """Compute azimuthally averaged 2D FFT power spectrum on a (B, C, H, W) tensor."""
    x = tensor_4d.float() - tensor_4d.float().mean(dim=(-2, -1), keepdim=True)
    fft = torch.fft.fft2(x, norm="ortho")
    fft_shift = torch.fft.fftshift(fft, dim=(-2, -1))
    power = (fft_shift.real ** 2 + fft_shift.imag ** 2).mean(dim=(0, 1)).cpu().numpy()
    H, W = power.shape
    cy, cx = H // 2, W // 2
    y, x_idx = np.indices((H, W))
    r = np.sqrt((x_idx - cx) ** 2 + (y - cy) ** 2).astype(int)
    tbin = np.bincount(r.ravel(), power.ravel())
    nr = np.bincount(r.ravel())
    radial = tbin / np.maximum(nr, 1)
    return radial[: H // 2]


@torch.no_grad()
def run_v3_trajectory_diagnostics(
    eval_model,
    vae,
    clip,
    prompt_list,
    guidance=3.0,
    num_steps=32,
    transition_sigma=None,
    unet_mask_gen=None,
    vit_mask_gen=None,
    mask_step=None,
):
    eval_model.eval()
    device = next(eval_model.parameters()).device
    if transition_sigma is None:
        transition_sigma = getattr(eval_model.net, "transition_sigma", 2.5)

    u_gen = unet_mask_gen if unet_mask_gen is not None else getattr(eval_model, "_eval_unet_mask_gen", None)
    v_gen = vit_mask_gen if vit_mask_gen is not None else getattr(eval_model, "_eval_vit_mask_gen", None)
    m_step = mask_step if mask_step is not None else getattr(eval_model, "_eval_mask_step", None)
    if u_gen is None or v_gen is None:
        u_gen, v_gen, m_step = build_eval_mask_generators(globals().get("active_cfg", {"transition_sigma": transition_sigma}))
    elif m_step is None:
        m_step = u_gen.total_steps

    if "ThermodynamicProbe" in globals() and hasattr(ThermodynamicProbe, "run_check"):
        ThermodynamicProbe(eval_model).run_check(res=32, device=device)

    pos_emb = clip.encode_text(prompt_list[:1])
    null_emb = clip.encode_text([""])

    step_indices = torch.arange(num_steps, dtype=torch.float32, device=device)
    s_min, s_max, rho = 0.002, 80.0, 7.0
    t_steps = (s_max ** (1 / rho) + step_indices / (num_steps - 1) * (s_min ** (1 / rho) - s_max ** (1 / rho))) ** rho
    t_steps = torch.cat([t_steps, torch.zeros_like(t_steps[:1])])

    x = torch.randn(1, 4, 32, 32, device=device) * t_steps[0]
    history = {"sigma": [], "vel_norm": [], "cos_sim": [], "w_vit": [], "w_unet": [], "gate": []}
    film_frames = []
    capture_steps = set(np.linspace(0, num_steps - 1, 8, dtype=int).tolist())

    def cfg_denoise(lat, sig):
        sig_t = sig.view(1, 1, 1, 1)
        sig_vec = sig.reshape(1).expand(lat.shape[0]).float()
        u_mask = u_gen(sigma=sig_vec, step=m_step)
        v_mask = v_gen(sigma=sig_vec, step=m_step)
        out_c = eval_model(
            x=lat, sigma=sig_t, text_emb=pos_emb,
            Unet_router_mask=u_mask, Vit_router_mask=v_mask, zeta=0.0,
        )
        if guidance == 1.0:
            return out_c["denoised"].float(), out_c
        out_u = eval_model(
            x=lat, sigma=sig_t, text_emb=null_emb,
            Unet_router_mask=u_mask, Vit_router_mask=v_mask, zeta=0.0,
        )
        d_cfg = out_u["denoised"].float().lerp(out_c["denoised"].float(), guidance)
        return d_cfg, out_c

    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])):
        d_cur_pred, out_c = cfg_denoise(x, t_cur)
        d_vec = (x.float() - d_cur_pred) / t_cur

        v_flat = out_c["vit_denoised"].float().flatten()
        u_flat = out_c["unet_denoised"].float().flatten()
        cos_uv = F.cosine_similarity(v_flat.unsqueeze(0), u_flat.unsqueeze(0)).item()

        history["sigma"].append(float(t_cur.item()))
        history["vel_norm"].append(float(d_vec.norm().item() / math.sqrt(d_vec.numel())))
        history["cos_sim"].append(float(cos_uv))
        history["w_vit"].append(float(out_c["w_vit"].mean().item()))
        history["w_unet"].append(float(out_c["w_unet"].mean().item()))
        history["gate"].append(float(out_c["gate_strength"].mean().item()))

        if i in capture_steps:
            dec = vae.decode(d_cur_pred).float().cpu() / 255.0
            film_frames.append((float(t_cur.item()), dec[0]))

        x_next = x + (t_next - t_cur) * d_vec
        if i < num_steps - 1:
            d_prime_pred, _ = cfg_denoise(x_next, t_next)
            d_prime = (x_next.float() - d_prime_pred) / t_next
            x = x + (t_next - t_cur) * (0.5 * d_vec + 0.5 * d_prime)
        else:
            x = x_next

    spec_real = None
    if "val_dataset" in globals() and len(val_dataset) >= 8:
        real_imgs = torch.stack([val_dataset[k]["pixel_values"] for k in range(8)]).to(device)
        real_lat = vae.encode(((real_imgs + 1.0) * 127.5).clamp(0, 255))
        spec_real = compute_radial_spectrum(real_lat)

    sampler = EDM_Sampler(
        eval_model,
        num_solve_steps=32,
        guidance=guidance,
        unet_mask_gen=u_gen,
        vit_mask_gen=v_gen,
        mask_step=m_step,
    )
    gen_embs = clip.encode_text(prompt_list[:4])
    gen_null = null_emb.expand(gen_embs.shape[0], -1, -1)
    gen_noise = torch.randn(gen_embs.shape[0], 4, 32, 32, device=device)
    gen_lats = sampler.sample(gen_noise, text_emb=gen_embs, uncond_text_emb=gen_null)
    spec_gen = compute_radial_spectrum(gen_lats)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    s_arr = history["sigma"]
    axes[0, 0].plot(s_arr, history["vel_norm"], marker="o", ms=3)
    axes[0, 0].set_xscale("log"); axes[0, 0].invert_xaxis()
    axes[0, 0].set_title("Lipschitz Velocity RMS ||(x - D_x)/sigma||")
    axes[0, 0].set_xlabel("sigma")

    axes[0, 1].plot(s_arr, history["cos_sim"], color="purple", marker="o", ms=3)
    axes[0, 1].set_xscale("log"); axes[0, 1].invert_xaxis()
    axes[0, 1].set_title("Branch Cosine Similarity (ViT vs U-Net D_x)")
    axes[0, 1].set_xlabel("sigma")

    axes[1, 0].plot(s_arr, history["w_vit"], label="w_vit")
    axes[1, 0].plot(s_arr, history["w_unet"], label="w_unet")
    axes[1, 0].plot(s_arr, history["gate"], "--", label="gate_strength")
    axes[1, 0].axvline(transition_sigma, color="gray", linestyle=":", label="transition_sigma")
    axes[1, 0].set_xscale("log"); axes[1, 0].invert_xaxis()
    axes[1, 0].set_title("Learned Blend & Transition Handoff Weights")
    axes[1, 0].legend()

    freqs = np.arange(len(spec_gen))
    ds_label = globals().get("ACTIVE_DATASET_NAME", "Validation")
    if spec_real is not None:
        axes[1, 1].semilogy(freqs, spec_real + 1e-12, label=f"Real {ds_label} Latents", linewidth=2)
    axes[1, 1].semilogy(freqs, spec_gen + 1e-12, label=f"Generated Latents (CFG={guidance})", linestyle="--")
    axes[1, 1].set_title("Latent Radial Power Spectrum (RAPSD)")
    axes[1, 1].set_xlabel("Spatial Frequency Radius (cycles / 32)")
    axes[1, 1].legend()
    plt.tight_layout()

    diag_dir = Path("/content/diagnostics")
    diag_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(diag_dir / "trajectory_diagnostics.png", dpi=200)
    plt.show()

    fig2, ax2 = plt.subplots(1, len(film_frames), figsize=(2.5 * len(film_frames), 3))
    for ax, (sig_v, frm) in zip(ax2, film_frames):
        ax.imshow(frm.permute(1, 2, 0).numpy().clip(0, 1))
        ax.set_title(f"sigma={sig_v:.2f}", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"Predicted Clean Trajectory: '{prompt_list[0]}'", fontsize=11)
    plt.tight_layout()
    fig2.savefig(diag_dir / "trajectory_filmstrip.png", dpi=200)
    plt.show()

    cfg_scales = [1.0, 3.0, 5.0]
    fixed_noise = torch.randn(len(prompt_list[:4]), 4, 32, 32, device=device)
    rows = []
    for g in cfg_scales:
        smp = EDM_Sampler(
            eval_model,
            num_solve_steps=32,
            guidance=g,
            unet_mask_gen=u_gen,
            vit_mask_gen=v_gen,
            mask_step=m_step,
        )
        lats = smp.sample(fixed_noise.clone(), text_emb=gen_embs, uncond_text_emb=gen_null)
        px = vae.decode(lats).float().cpu() / 255.0
        rows.append(px)
    grid = make_grid(torch.cat(rows, dim=0), nrow=4, padding=4)
    save_image(grid, diag_dir / "cfg_sweep_samples.png")

    fig3, ax3 = plt.subplots(figsize=(14, 10))
    ax3.imshow(grid.permute(1, 2, 0).numpy().clip(0, 1))
    ax3.set_title("Top->Bottom: CFG 1.0, 3.0, 5.0 | Prompts: " + " | ".join(prompt_list[:4]), fontsize=10)
    ax3.axis("off")
    plt.tight_layout()
    plt.show()

    if globals().get("USE_GOOGLE_DRIVE", False):
        drive_diag = globals().get("DRIVE_RUN_DIR", Path("/content/drive/MyDrive/HDMOEM_colab") / globals().get("RUN_NAME", "hdmoem_v3_nature50k_256_v3")) / "diagnostics"
        if "sync_to_drive" in globals():
            sync_to_drive(str(diag_dir), drive_diag)
        else:
            drive_diag.mkdir(parents=True, exist_ok=True)
            for item in diag_dir.iterdir():
                if item.is_file():
                    shutil.copy2(item, drive_diag / item.name)


eval_device = "cuda" if torch.cuda.is_available() else "cpu"
if eval_device == "cuda":
    torch.cuda.empty_cache()

eval_ckpt_path = resolve_final_checkpoint(CKPT_PATH)
eval_model, active_cfg = load_eval_model_from_checkpoint(eval_ckpt_path, use_ema=USE_EMA_WEIGHTS, device=eval_device)

if "clip_embedder" not in globals() or clip_embedder is None:
    clip_embedder = CLIP_EMBED(device=eval_device)
    clip_embedder.init()

if "vae_verified" not in globals() or vae_verified is None:
    vae_verified = StabilityVAE(
        raw_mean=active_cfg["raw_mean"],
        raw_std=active_cfg["raw_std"],
        final_mean=active_cfg.get("mean_data", 0.0),
        final_std=active_cfg["sigma_data"],
    )
    vae_verified.init(device=eval_device)

if "val_dataset" in globals() and len(val_dataset) >= 43:
    sample_prompts = [
        val_dataset[0]["prompts"],
        val_dataset[10]["prompts"],
        val_dataset[25]["prompts"],
        val_dataset[42]["prompts"],
    ]
else:
    sample_prompts = [
        "dramatic snow-capped mountain peaks reflected in a calm alpine lake at sunrise",
        "sunlight filtering through a dense misty pine forest onto a mossy stream",
        "turquoise ocean waves crashing against rugged coastal cliffs under golden clouds",
        "a cascading waterfall surrounded by lush green ferns and wet boulders",
    ]

print("Evaluation Prompts:")
for idx, pr in enumerate(sample_prompts):
    print(f"  [{idx}] {pr}")

run_v3_trajectory_diagnostics(
    eval_model=eval_model,
    vae=vae_verified,
    clip=clip_embedder,
    prompt_list=sample_prompts,
    guidance=3.0,
    num_steps=32,
    transition_sigma=active_cfg["transition_sigma"],
)


## 9. Quantitative FID & CLIP-Score Evaluation (`Utils.evaluator`)

Computes standard Frechet Inception Distance (and optional CLIPScore) on validation prompts
across CFG scales, caching real `val_dataset` Inception statistics to Drive for fast re-runs.


In [ ]:
# @title 9. Evaluate Quantitative FID & CLIP-Score (`Utils.evaluator`)
import json
from pathlib import Path
import torch
from Utils.evaluator import evaluate_model_fid
from Utils.VAE_CLIP import CLIP_EMBED, StabilityVAE

FID_NUM_SAMPLES = 5000     # @param {type:"integer"}  # 5000 matches COCO val2017 FID-5k
FID_BATCH_SIZE = 32        # @param {type:"integer"}
FID_NUM_STEPS = 32         # @param {type:"integer"}
FID_CFG_SCALES = [1.0, 2.0, 3.0]
COMPUTE_CLIP_SCORE = True  # @param {type:"boolean"}
SAVE_EVAL_IMAGES = False   # @param {type:"boolean"}

eval_device = "cuda" if torch.cuda.is_available() else "cpu"
eval_ckpt_path = resolve_final_checkpoint(CKPT_PATH if "CKPT_PATH" in globals() else "")
eval_model, eval_cfg = load_eval_model_from_checkpoint(
    eval_ckpt_path,
    use_ema=globals().get("USE_EMA_WEIGHTS", True),
    device=eval_device,
)

if "clip_embedder" not in globals() or clip_embedder is None:
    clip_embedder = CLIP_EMBED(device=eval_device)
    clip_embedder.init()

vae_for_fid = StabilityVAE(
    raw_mean=eval_cfg["raw_mean"],
    raw_std=eval_cfg["raw_std"],
    final_mean=eval_cfg.get("mean_data", 0.0),
    final_std=eval_cfg["sigma_data"],
)
vae_for_fid.init(device=eval_device)

ds_tag = globals().get("DATASET_MODE", globals().get("ACTIVE_DATASET_NAME", "custom"))
base_drive = globals().get("DRIVE_BASE_DIR", Path("/content/drive/MyDrive/HDMOEM_colab"))
stats_cache_file = base_drive / "fid_stats" / f"{ds_tag}_{eval_cfg.get('data_img_res', 256)}_stats.pt"
save_gen_dir = (Path("/content/fid_generated") / eval_cfg["run_name"]) if SAVE_EVAL_IMAGES else None

fid_report = evaluate_model_fid(
    model=eval_model,
    vae=vae_for_fid,
    clip=clip_embedder,
    val_dataset=val_dataset,
    real_dataset=val_dataset,
    model_config=eval_cfg,
    mask_config=base_mask_configs,
    dataset_name=ds_tag,
    guidance_scales=FID_CFG_SCALES,
    num_eval_samples=FID_NUM_SAMPLES,
    num_solve_steps=FID_NUM_STEPS,
    batch_size=FID_BATCH_SIZE,
    seed=1234,
    save_generated_dir=save_gen_dir,
    stats_cache_path=stats_cache_file,
    fid_backend="auto",
    device=eval_device,
    use_amp=True,
    compute_clip_score=COMPUTE_CLIP_SCORE,
)

out_json = Path("/content") / f"{eval_cfg['run_name']}_fid_report.json"
out_json.write_text(json.dumps(fid_report, indent=2), encoding="utf-8")
print(f"Saved FID report -> {out_json}")
if globals().get("USE_GOOGLE_DRIVE", False):
    drive_eval_dir = DRIVE_RUN_DIR / "eval"
    drive_eval_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(out_json, drive_eval_dir / out_json.name)
